<a href="https://colab.research.google.com/github/Pratyushan4191/Axiom_Microplastics/blob/main/Axiom_microplastics_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!rm -rf /content/drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!find /content/drive/MyDrive -maxdepth 3

#microplastic

Step 1 — create the experiment configuration

In [ ]:
# ============================================================
# MICROPLASTIC SEGMENTATION
# STEP 1 — LOCK BASELINE EXPERIMENT CONFIGURATION
# ============================================================

from pathlib import Path
import json
import random
import numpy as np
import torch

PROJECT_ROOT = Path("/content/drive/MyDrive/MicroPlastics/research_workspace")

ANALYSIS_DIR = PROJECT_ROOT / "analysis"
BEACH_ROOT = PROJECT_ROOT / "extracted" / "beach"
WWTP_ROOT = PROJECT_ROOT / "unified" / "wwtp"
DEEPPARTICLE_ROOT = PROJECT_ROOT / "extracted" / "deepparticle"

BEACH_MANIFEST = ANALYSIS_DIR / "beach_segmentation_manifest.csv"
BEACH_CLEAN_MANIFEST = ANALYSIS_DIR / "beach_clean_manifest.csv"
WWTP_MANIFEST = ANALYSIS_DIR / "wwtp_canonical_manifest.csv"
UNIFIED_MANIFEST = ANALYSIS_DIR / "microplastic_final_unified_manifest.csv"

EXPERIMENT_DIR = ANALYSIS_DIR / "baseline_experiment"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "experiment_name": "beach_supervised_unet_resnet50_baseline",

    "task": {
        "type": "binary_semantic_segmentation",
        "input_channels": 3,
        "output_channels": 1,
        "input_size": [512, 512],
    },

    "dataset": {
        "train_domain": "Beach",
        "validation_domain": "Beach",
        "primary_test_domain": "Beach",
        "external_test_domain": "WWTP",

        "train_target_status": ["annotated_positive"],
        "validation_target_status": ["annotated_positive"],

        "primary_test_status": [
            "test_positive"
        ],

        "empty_mask_policy": {
            "status": "annotation_empty",
            "confirmed_negative": False,
            "used_for_training": False,
            "used_for_validation": False,
            "used_for_primary_metric": False,
            "retained_for_secondary_analysis": True,
        },

        "wwtp_training": False,
        "wwtp_validation": False,
        "wwtp_model_selection": False,
        "wwtp_external_test": True,
    },

    "model": {
        "architecture": "UNet",
        "encoder": "ResNet50",
        "encoder_weights": "imagenet",
        "activation": None,
    },

    "loss": {
        "type": "BCEWithLogitsPlusDice",
        "bce_weight": 0.5,
        "dice_weight": 0.5,
    },

    "optimizer": {
        "type": "AdamW",
        "learning_rate": 1e-4,
        "weight_decay": 1e-4,
    },

    "training": {
        "batch_size": 8,
        "epochs": 50,
        "early_stopping_patience": 10,
        "num_workers": 2,
        "pin_memory": True,
        "mixed_precision": True,
    },

    "augmentation": {
        "horizontal_flip_p": 0.5,
        "rotation_degrees": 10,
        "scale": [0.95, 1.05],
        "translation": [0.05, 0.05],
        "brightness": [0.9, 1.1],
        "contrast": [0.9, 1.1],
    },

    "evaluation": {
        "threshold": 0.5,
        "metrics": [
            "dice",
            "iou",
            "precision",
            "recall",
            "pixel_accuracy",
        ],
    },

    "reproducibility": {
        "seed": 42,
    },

    "wwtp_tiling": {
        "status": "TO_BE_LOCKED_BEFORE_EXTERNAL_TEST",
        "native_image_size": [2160, 3840],
        "tile_size": [512, 512],
        "stride": None,
        "overlap": None,
        "stitching": "TO_BE_LOCKED",
        "edge_handling": "TO_BE_LOCKED",
    },

    "research_hierarchy": {
        "primary": "Beach supervised -> WWTP external",
        "secondary": "DeepParticle weak supervision/pretraining",
        "later": "augmentation robustness",
        "later": "domain generalization/adaptation",
    },
}

config_path = EXPERIMENT_DIR / "baseline_config.json"

with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Configuration written to:")
print(config_path)
print()
print(json.dumps(CONFIG, indent=2))

In [ ]:
# ============================================================
# STEP 2 — VERIFY DATA ARTIFACTS
# ============================================================

required_files = {
    "Beach segmentation manifest": BEACH_MANIFEST,
    "Beach clean manifest": BEACH_CLEAN_MANIFEST,
    "WWTP manifest": WWTP_MANIFEST,
    "Unified manifest": UNIFIED_MANIFEST,
}

for name, path in required_files.items():
    print(f"{name:30s} : {'OK' if path.exists() else 'MISSING'}")
    if path.exists():
        print(f"    {path}")

assert all(p.exists() for p in required_files.values()), \
    "One or more required manifests are missing."

print("\nAll required manifests are present.")

In [ ]:
import pandas as pd

beach = pd.read_csv(BEACH_MANIFEST)

print("Columns:")
print(beach.columns.tolist())

print("\nShape:")
print(beach.shape)

print("\nRole counts:")
print(beach["segmentation_role"].value_counts(dropna=False))

print("\nTarget-status counts:")
if "target_status" in beach.columns:
    print(beach["target_status"].value_counts(dropna=False))

print("\nSplit/domain counts:")
for col in ["domain", "split", "segmentation_role"]:
    if col in beach.columns:
        print(f"\n{col}:")
        print(beach[col].value_counts(dropna=False))

print("\nFirst records:")
display(beach.head())

In [ ]:
# ============================================================
# STEP 4 — EMPTY MASK SEMANTICS DECISION
# ============================================================

empty_policy = {
    "Beach_empty_masks": 440,
    "WWTP_empty_images": 6,

    "confirmed_negative": False,

    "reason": (
        "The available dataset annotations establish that these images "
        "have empty masks/annotations, but the audited metadata does not "
        "establish that they are confirmed true-negative samples."
    ),

    "training": "EXCLUDE",
    "validation": "EXCLUDE",
    "primary_test_metrics": "EXCLUDE",
    "secondary_analysis": "RETAIN",

    "future_resolution": [
        "Inspect original dataset documentation/provenance.",
        "Determine whether empty annotations explicitly mean no target.",
        "If confirmed negatives, they may be incorporated into a defined negative evaluation.",
        "If not confirmed, retain them as annotation-empty/unknown."
    ],
}

empty_policy_path = EXPERIMENT_DIR / "empty_mask_semantics.json"

with open(empty_policy_path, "w") as f:
    json.dump(empty_policy, f, indent=2)

print(json.dumps(empty_policy, indent=2))

In [ ]:
# ============================================================
# STEP 5 — VERIFY EXCLUDED CROSS-SPLIT EXACT DUPLICATES
# ============================================================

clean_beach = pd.read_csv(BEACH_CLEAN_MANIFEST)

print("Clean Beach manifest shape:", clean_beach.shape)

if "evaluation_eligible" in clean_beach.columns:
    print("\nEvaluation eligibility:")
    print(clean_beach["evaluation_eligible"].value_counts(dropna=False))

print("\nTest records:")
if "role" in clean_beach.columns:
    print(clean_beach["role"].value_counts(dropna=False))

print("\nExpected primary test:")
print("42 annotated-positive test images")
print("17 annotation-empty test images")
print("2 exact train/test duplicates excluded from evaluation")

In [ ]:
# ============================================================
# STEP 6A — INSPECT BEACH FILENAME STRUCTURE
# ============================================================

import os
import re

all_beach = pd.concat(
    [
        pd.read_csv(BEACH_MANIFEST),
    ],
    ignore_index=True
)

image_paths = all_beach["image_path"].astype(str)

print("Example filenames:")
for p in image_paths.head(30):
    print(Path(p).name)

In [ ]:
import os
import re
import pandas as pd # Ensure pandas is imported
from pathlib import Path # Ensure Path is imported

# ============================================================
# STEP 6B — CANDIDATE PARENT-ID EXTRACTION
# ============================================================

def candidate_parent_id(filename):
    """
    Conservative filename parsing.

    This does NOT claim that the extracted value is the true
    parent photograph ID. It only creates candidate groups
    that must subsequently be audited.
    """
    stem = Path(filename).stem

    # Remove common patch/augmentation suffixes.
    x = re.sub(r"_part_\d+$", "", stem, flags=re.I)
    x = re.sub(r"_partition_\d+$", "", x, flags=re.I)
    x = re.sub(r"_augmented$", "", x, flags=re.I)

    return x


all_beach["candidate_parent_id"] = [
    candidate_parent_id(Path(p).name)
    for p in all_beach["image_path"]
]

cross_split_groups = (
    all_beach
    .groupby("candidate_parent_id")
    .agg(
        n=("image_path", "count"),
        splits=("segmentation_role", lambda x: sorted(set(x))), # Changed 'role' to 'segmentation_role'
        files=("image_path", list),
    )
    .reset_index()
)

cross_split_groups = cross_split_groups[
    cross_split_groups["splits"].apply(len) > 1
]

print("Candidate parent groups crossing roles:")
print(len(cross_split_groups))

display(cross_split_groups.head(50))

In [ ]:
train_df = beach[beach["segmentation_role"] == "train"].copy()
val_df = beach[beach["segmentation_role"] == "val"].copy()
test_positive_df = beach[beach["segmentation_role"] == "test_positive"].copy()
test_empty_df = beach[beach["segmentation_role"] == "test_empty"].copy()

assert len(train_df) == 1041
assert len(val_df) == 45
assert len(test_positive_df) == 42
assert len(test_empty_df) == 17

print("TRAIN:", len(train_df))
print("VAL:", len(val_df))
print("TEST POSITIVE:", len(test_positive_df))
print("TEST EMPTY:", len(test_empty_df))

split_lock = {
    "train": len(train_df),
    "validation": len(val_df),
    "primary_test_positive": len(test_positive_df),
    "secondary_test_empty": len(test_empty_df),
}

with open(EXPERIMENT_DIR / "beach_split_lock.json", "w") as f:
    json.dump(split_lock, f, indent=2)

print("\nBeach split locked.")

Step 8 — perform a definitive Beach parent-group audit

In [ ]:
# ============================================================
# STEP 8 — DEFINITIVE BEACH PARENT-GROUP AUDIT
# ============================================================

import pandas as pd
from pathlib import Path
import re

beach = pd.read_csv(BEACH_MANIFEST).copy()


def infer_parent_group(filename):
    """
    Infer the likely parent source image from the Beach patch filename.

    Examples:
        0_part_1.png       -> 0
        0_part_2.png       -> 0
        31_part_2.png      -> 31
        DSC_0320_...       -> DSC_0320...
        1000_augmented.png -> 1000

    This is a provenance heuristic, not yet a ground-truth
    source-photo identifier.
    """
    stem = Path(filename).stem

    # Remove patch suffix.
    stem = re.sub(r"_part_\d+$", "", stem, flags=re.IGNORECASE)

    # Remove augmentation suffix.
    stem = re.sub(r"_augmented$", "", stem, flags=re.IGNORECASE)

    # Remove partition suffix if present.
    stem = re.sub(r"_partition_\d+$", "", stem, flags=re.IGNORECASE)

    return stem


beach["parent_group"] = beach["image"].apply(infer_parent_group)

# ------------------------------------------------------------
# Parent-group summary
# ------------------------------------------------------------

parent_summary = (
    beach
    .groupby("parent_group")
    .agg(
        n_images=("image", "size"),
        roles=("segmentation_role",
               lambda x: sorted(set(x))),
        splits=("split",
                lambda x: sorted(set(x))),
        images=("image", list),
    )
    .reset_index()
)

parent_summary["n_roles"] = parent_summary["roles"].apply(len)

# Groups spanning multiple experimental roles.
cross_role = parent_summary[
    parent_summary["n_roles"] > 1
].copy()

print("Total parent groups:", len(parent_summary))
print("Parent groups crossing roles:", len(cross_role))

display(
    cross_role[
        [
            "parent_group",
            "n_images",
            "roles",
            "splits",
            "images",
        ]
    ].head(100)
)

# Save complete audit.
parent_audit_path = (
    EXPERIMENT_DIR / "beach_parent_group_audit.csv"
)

parent_summary.to_csv(parent_audit_path, index=False)

cross_role_path = (
    EXPERIMENT_DIR / "beach_cross_role_parent_groups.csv"
)

cross_role.to_csv(cross_role_path, index=False)

print("\nSaved:")
print(parent_audit_path)
print(cross_role_path)

In [ ]:
# ============================================================
# STEP 9 — CROSS-ROLE PARENT GROUP COUNTS
# ============================================================

def has_role(group, role):
    return role in group["roles"]

cross_role["has_train"] = cross_role["roles"].apply(
    lambda x: "train" in x
)

cross_role["has_val"] = cross_role["roles"].apply(
    lambda x: "val" in x
)

cross_role["has_test_positive"] = cross_role["roles"].apply(
    lambda x: "test_positive" in x
)

cross_role["has_test_empty"] = cross_role["roles"].apply(
    lambda x: "test_empty" in x
)

print(
    "Train + Val:",
    (
        cross_role.has_train &
        cross_role.has_val
    ).sum()
)

print(
    "Train + Test-positive:",
    (
        cross_role.has_train &
        cross_role.has_test_positive
    ).sum()
)

print(
    "Val + Test-positive:",
    (
        cross_role.has_val &
        cross_role.has_test_positive
    ).sum()
)

print(
    "Train + Val + Test-positive:",
    (
        cross_role.has_train &
        cross_role.has_val &
        cross_role.has_test_positive
    ).sum()
)

print(
    "Train + Test-empty:",
    (
        cross_role.has_train &
        cross_role.has_test_empty
    ).sum()
)

print(
    "Test-positive + Test-empty:",
    (
        cross_role.has_test_positive &
        cross_role.has_test_empty
    ).sum()
)

In [ ]:
# ============================================================
# STEP 10 — AFFECTED IMAGE COUNT
# ============================================================

cross_parent_ids = set(
    cross_role["parent_group"]
)

affected = beach[
    beach["parent_group"].isin(cross_parent_ids)
].copy()

print("Cross-role parent groups:", len(cross_parent_ids))
print("Affected image records:", len(affected))

print("\nAffected images by role:")
print(
    affected["segmentation_role"]
    .value_counts()
)

print("\nAffected images by split:")
print(
    affected["split"]
    .value_counts()
)

In [ ]:
# ============================================================
# STEP 11 — SHOW ALL CROSS-ROLE GROUPS
# ============================================================

pd.set_option("display.max_colwidth", 100)

for _, row in cross_role.iterrows():

    print("=" * 80)
    print("PARENT GROUP:", row["parent_group"])
    print("N IMAGES:", row["n_images"])
    print("ROLES:", row["roles"])
    print("SPLITS:", row["splits"])

    group_rows = beach[
        beach["parent_group"] == row["parent_group"]
    ][
        [
            "image",
            "split",
            "segmentation_role",
            "mask_status",
            "positive_fraction",
            "evaluation_eligible",
            "exclusion_reason",
        ]
    ]

    display(group_rows)

In [ ]:
# ============================================================
# STEP 13 — CREATE LEAKAGE-FREE PARENT-LEVEL SPLIT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split


# ------------------------------------------------------------
# Load complete Beach segmentation manifest
# ------------------------------------------------------------

beach = pd.read_csv(BEACH_MANIFEST).copy()

# Recreate the parent group from the audited filename logic.
# IMPORTANT: use the same grouping rule that produced the
# validated 817 parent groups.

import re

def infer_parent_group(filename):
    stem = Path(filename).stem

    stem = re.sub(
        r"_part_\d+$",
        "",
        stem,
        flags=re.IGNORECASE
    )

    stem = re.sub(
        r"_augmented$",
        "",
        stem,
        flags=re.IGNORECASE
    )

    stem = re.sub(
        r"_partition_\d+$",
        "",
        stem,
        flags=re.IGNORECASE
    )

    return stem


beach["parent_group"] = beach["image"].apply(
    infer_parent_group
)


# ------------------------------------------------------------
# Parent-level summary
# ------------------------------------------------------------

parent_table = (
    beach
    .groupby("parent_group")
    .agg(
        n_images=("image", "size"),

        n_positive=(
            "mask_status",
            lambda x: (x == "annotated_positive").sum()
        ),

        n_empty=(
            "mask_status",
            lambda x: (x == "annotation_empty").sum()
        ),

        positive_fraction_mean=(
            "positive_fraction",
            "mean"
        ),

        positive_fraction_max=(
            "positive_fraction",
            "max"
        ),
    )
    .reset_index()
)

print("Parent groups:", len(parent_table))
print(parent_table["n_images"].describe())

In [ ]:
# ------------------------------------------------------------
# 70 / 15 / 15 parent-level split
# ------------------------------------------------------------

SEED = 42

parents = parent_table["parent_group"].to_numpy()

train_parents, temp_parents = train_test_split(
    parents,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
)

val_parents, test_parents = train_test_split(
    temp_parents,
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
)

train_parents = set(train_parents)
val_parents = set(val_parents)
test_parents = set(test_parents)

print("Train parents:", len(train_parents))
print("Validation parents:", len(val_parents))
print("Test parents:", len(test_parents))

assert train_parents.isdisjoint(val_parents)
assert train_parents.isdisjoint(test_parents)
assert val_parents.isdisjoint(test_parents)

assert (
    len(train_parents)
    + len(val_parents)
    + len(test_parents)
    == len(parent_table)
)

print("Parent-level split is mutually exclusive.")

In [ ]:
# ============================================================
# STEP 14 — CREATE IMMUTABLE GROUPED BASELINE MANIFEST
# ============================================================

def assign_split(parent):
    if parent in train_parents:
        return "train"
    elif parent in val_parents:
        return "val"
    elif parent in test_parents:
        return "test"
    else:
        raise RuntimeError(
            f"Parent group not assigned: {parent}"
        )


beach["baseline_split"] = beach["parent_group"].apply(
    assign_split
)

beach["group_assignment"] = (
    "parent_level_seed_42"
)

# ------------------------------------------------------------
# Operational roles
# ------------------------------------------------------------

def baseline_role(row):

    if row["baseline_split"] == "train":
        if row["mask_status"] == "annotated_positive":
            return "train"

        # Empty masks are retained in the manifest but excluded
        # from supervised training.
        return "train_annotation_empty"

    if row["baseline_split"] == "val":
        if row["mask_status"] == "annotated_positive":
            return "val"

        return "val_annotation_empty"

    if row["baseline_split"] == "test":
        if row["mask_status"] == "annotated_positive":
            return "test_positive"

        return "test_empty"

    raise RuntimeError()


beach["baseline_role"] = beach.apply(
    baseline_role,
    axis=1
)

# ------------------------------------------------------------
# Evaluation eligibility
# ------------------------------------------------------------

beach["baseline_evaluation_eligible"] = (
    beach["baseline_role"] == "test_positive"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

GROUPED_MANIFEST = (
    EXPERIMENT_DIR /
    "beach_baseline_grouped_manifest.csv"
)

beach.to_csv(
    GROUPED_MANIFEST,
    index=False
)

print("Saved:")
print(GROUPED_MANIFEST)

In [ ]:
# ============================================================
# STEP 15 — VERIFY ZERO PARENT LEAKAGE
# ============================================================

check = (
    beach
    .groupby("parent_group")["baseline_split"]
    .nunique()
)

bad_groups = check[check > 1]

print("Parent groups:", len(check))
print("Groups crossing baseline splits:", len(bad_groups))

assert len(bad_groups) == 0, (
    "LEAKAGE DETECTED: some parent groups still span "
    "multiple baseline splits."
)

print("PASS: zero parent groups cross train/val/test.")

In [ ]:
# ============================================================
# STEP 16 — NEW GROUPED SPLIT COUNTS
# ============================================================

print("\nIMAGE COUNTS:")
print(
    beach["baseline_split"]
    .value_counts()
    .sort_index()
)

print("\nROLE COUNTS:")
print(
    beach["baseline_role"]
    .value_counts()
    .sort_index()
)

print("\nPARENT COUNTS:")
print(
    beach.groupby("baseline_split")["parent_group"]
    .nunique()
    .sort_index()
)

print("\nMASK STATUS × SPLIT:")
print(
    pd.crosstab(
        beach["baseline_split"],
        beach["mask_status"]
    )
)

In [ ]:
# ============================================================
# STEP 17 — FOREGROUND DISTRIBUTION BY GROUPED SPLIT
# ============================================================

positive_only = beach[
    beach["mask_status"] == "annotated_positive"
].copy()

summary = (
    positive_only
    .groupby("baseline_split")["positive_fraction"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
)

display(summary)

In [ ]:
# ============================================================
# STEP 6 — BEACH BASELINE DATASET / DATALOADER
# ============================================================

import os
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

MANIFEST_PATH = (
    "/content/drive/MyDrive/MicroPlastics/"
    "research_workspace/analysis/baseline_experiment/"
    "beach_baseline_grouped_manifest.csv"
)

df = pd.read_csv(MANIFEST_PATH)

print("Manifest shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nBaseline split counts:")
print(df["baseline_split"].value_counts())

print("\nBaseline role counts:")
print(df["baseline_role"].value_counts())

print("\nMask status × split:")
print(
    pd.crosstab(
        df["baseline_split"],
        df["mask_status"]
    )
)


# ------------------------------------------------------------
# 2. Verify the immutable split
# ------------------------------------------------------------

assert df["parent_group"].notna().all(), \
    "Missing parent_group values."

crossing = (
    df.groupby("parent_group")["baseline_split"]
      .nunique()
)

assert crossing.max() == 1, \
    "Parent leakage detected: a parent occurs in multiple splits."

print("\nPASS: no parent group crosses train/val/test.")


# ------------------------------------------------------------
# 3. Dataset
# ------------------------------------------------------------

class BeachSegmentationDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_size=(512, 512),
        training=False
    ):
        self.df = dataframe.reset_index(drop=True)
        self.image_size = image_size
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = row["image_path"]
        mask_path = row["mask_path"]

        # -------------------------
        # Read image
        # -------------------------
        image = Image.open(image_path).convert("RGB")

        # -------------------------
        # Read mask
        # -------------------------
        mask = Image.open(mask_path).convert("L")

        # -------------------------
        # Convert to tensors
        # -------------------------
        image = TF.to_tensor(image)

        mask = np.array(mask, dtype=np.uint8)

        # Strict binary conversion:
        # 0 -> background
        # 255 -> foreground
        mask = (mask > 0).astype(np.float32)

        mask = torch.from_numpy(mask).unsqueeze(0)

        # -------------------------
        # Resize
        # -------------------------
        image = TF.resize(
            image,
            self.image_size,
            interpolation=InterpolationMode.BILINEAR,
            antialias=True
        )

        mask = TF.resize(
            mask,
            self.image_size,
            interpolation=InterpolationMode.NEAREST
        )

        # Make absolutely binary after resize
        mask = (mask > 0.5).float()

        # -------------------------
        # Training augmentation
        # -------------------------
        if self.training:

            if random.random() < 0.5:
                image = torch.flip(image, dims=[2])
                mask = torch.flip(mask, dims=[2])

            # Small rotation
            angle = random.uniform(-10.0, 10.0)

            image = TF.rotate(
                image,
                angle,
                interpolation=InterpolationMode.BILINEAR,
                fill=0
            )

            mask = TF.rotate(
                mask,
                angle,
                interpolation=InterpolationMode.NEAREST,
                fill=0
            )

            mask = (mask > 0.5).float()

        return {
            "image": image.float(),
            "mask": mask.float(),
            "image_path": image_path,
            "mask_path": mask_path,
            "sample_id": str(row.get("image", idx)),
            "mask_status": row["mask_status"],
            "parent_group": str(row["parent_group"]),
            "baseline_split": row["baseline_split"],
        }


# ------------------------------------------------------------
# 4. Create the supervised subsets
# ------------------------------------------------------------

train_df = df[
    (df["baseline_split"] == "train") &
    (df["mask_status"] == "annotated_positive")
].copy()

val_df = df[
    (df["baseline_split"] == "val") &
    (df["mask_status"] == "annotated_positive")
].copy()

test_df = df[
    (df["baseline_split"] == "test") &
    (df["mask_status"] == "annotated_positive")
].copy()

test_empty_df = df[
    (df["baseline_split"] == "test") &
    (df["mask_status"] == "annotation_empty")
].copy()


print("\nSUPERVISED DATASETS")
print("-------------------")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))
print("Test annotation-empty:", len(test_empty_df))


# ------------------------------------------------------------
# 5. Create Dataset objects
# ------------------------------------------------------------

train_dataset = BeachSegmentationDataset(
    train_df,
    image_size=(512, 512),
    training=True
)

val_dataset = BeachSegmentationDataset(
    val_df,
    image_size=(512, 512),
    training=False
)

test_dataset = BeachSegmentationDataset(
    test_df,
    image_size=(512, 512),
    training=False
)


# ------------------------------------------------------------
# 6. DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)


# ------------------------------------------------------------
# 7. Pull one batch
# ------------------------------------------------------------

batch = next(iter(train_loader))

print("\nBATCH VERIFICATION")
print("------------------")
print("Image shape:", batch["image"].shape)
print("Mask shape:", batch["mask"].shape)
print("Image dtype:", batch["image"].dtype)
print("Mask dtype:", batch["mask"].dtype)

print("Image min:", batch["image"].min().item())
print("Image max:", batch["image"].max().item())

print("Mask unique values:",
      torch.unique(batch["mask"]).tolist())

print("Mask positive fraction:",
      batch["mask"].mean().item())


# ------------------------------------------------------------
# 8. Hard assertions
# ------------------------------------------------------------

assert batch["image"].ndim == 4
assert batch["mask"].ndim == 4

assert batch["image"].shape[1:] == (3, 512, 512)
assert batch["mask"].shape[1:] == (1, 512, 512)

assert batch["image"].dtype == torch.float32
assert batch["mask"].dtype == torch.float32

mask_values = torch.unique(batch["mask"]).cpu().numpy()

assert set(mask_values.tolist()).issubset({0.0, 1.0}), \
    f"Mask is not binary: {mask_values}"

assert torch.isfinite(batch["image"]).all()
assert torch.isfinite(batch["mask"]).all()

print("\n===================================")
print("STEP 6 DATASET/DATALOADER: PASS")
print("===================================")

In [ ]:
# ============================================================
# STEP 7 — VISUALIZE IMAGE + MASK
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

def visualize_batch(batch, n=4):

    n = min(n, batch["image"].shape[0])

    fig, axes = plt.subplots(
        n,
        3,
        figsize=(12, 4 * n)
    )

    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(n):

        image = batch["image"][i].permute(1, 2, 0).cpu().numpy()
        mask = batch["mask"][i, 0].cpu().numpy()

        axes[i, 0].imshow(image)
        axes[i, 0].set_title("RGB image")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(mask)
        axes[i, 1].set_title("Ground-truth mask")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(image)
        axes[i, 2].imshow(mask, alpha=0.35)
        axes[i, 2].set_title("Overlay")
        axes[i, 2].axis("off")

    plt.tight_layout()
    plt.show()


visualize_batch(batch, n=4)

In [ ]:
# ============================================================
# STEP 7B — LOCK WWTP AS EXTERNAL EVALUATION DOMAIN
# ============================================================

import os
import json
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BASE = "/content/drive/MyDrive/MicroPlastics/research_workspace"

WWTP_MANIFEST = os.path.join(
    BASE, "analysis", "wwtp_canonical_manifest.csv"
)

OUT_DIR = os.path.join(
    BASE, "analysis", "baseline_experiment"
)

os.makedirs(OUT_DIR, exist_ok=True)

OUT_MANIFEST = os.path.join(
    OUT_DIR, "wwtp_external_evaluation_manifest.csv"
)

OUT_CONFIG = os.path.join(
    OUT_DIR, "wwtp_external_evaluation_config.json"
)

# ------------------------------------------------------------
# 2. Load canonical WWTP manifest
# ------------------------------------------------------------

df = pd.read_csv(WWTP_MANIFEST)

print("Original WWTP manifest shape:", df.shape)
print("\nOriginal columns:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 3. Verify expected dataset size
# ------------------------------------------------------------

assert len(df) == 2010, (
    f"Expected 2010 WWTP records, found {len(df)}"
)

# ------------------------------------------------------------
# 4. Preserve original dataset split
# ------------------------------------------------------------

print("\nOriginal WWTP split:")
print(df["split"].value_counts(dropna=False))

# Expected:
# train        1406
# validation    302
# test          302

# ------------------------------------------------------------
# 5. Assign project-level external evaluation role
# ------------------------------------------------------------

df["external_evaluation"] = True
df["external_role"] = "wwtp_external"

# The original train/validation/test labels are retained only
# as provenance. They are NOT used for model training,
# validation, checkpoint selection, or threshold selection.

df["project_split_role"] = "external_evaluation"

# ------------------------------------------------------------
# 6. Define primary metric eligibility
# ------------------------------------------------------------

# Existing canonical manifest contains mask_status instead of target_status.
print("\nMask status distribution:")
if "mask_status" in df.columns:
    print(df["mask_status"].value_counts(dropna=False))
else:
    print("mask_status column not found.")

# Primary evaluation:
#   annotated_positive -> included
#
# Secondary:
#   annotation_empty -> retained but excluded from primary metrics

if "mask_status" in df.columns:
    df["primary_metric_eligible"] = (
        df["mask_status"] == "annotated_positive"
    )
else:
    raise ValueError(
        "Expected 'mask_status' column in WWTP canonical manifest."
    )

df["secondary_analysis_eligible"] = (
    df["mask_status"] == "annotation_empty"
)

# ------------------------------------------------------------
# 7. Add explicit reason for metric exclusion
# ------------------------------------------------------------

df["primary_metric_exclusion_reason" ] = ""

df.loc[
    df["mask_status"] == "annotation_empty",
    "primary_metric_exclusion_reason"
] = "annotation_empty_not_confirmed_negative"

# ------------------------------------------------------------
# 8. Verify expected populations
# ------------------------------------------------------------

print("\nProject-level evaluation population:")

print(
    "Total WWTP images:",
    len(df)
)

print(
    "Primary metric eligible:",
    int(df["primary_metric_eligible"].sum())
)

print(
    "Secondary annotation-empty:",
    int(df["secondary_analysis_eligible"].sum())
)

# Expected:
# Total = 2010
# Primary = 2004
# Empty = 6

assert len(df) == 2010
assert df["primary_metric_eligible"].sum() == 2004
assert df["secondary_analysis_eligible"].sum() == 6

# ------------------------------------------------------------
# 9. Verify no WWTP image is assigned to training/validation
# ------------------------------------------------------------

assert (df["project_split_role"] == "external_evaluation").all()
assert (df["external_evaluation"] == True).all()

# ------------------------------------------------------------
# 10. Save external evaluation manifest
# ------------------------------------------------------------

df.to_csv(OUT_MANIFEST, index=False)

print("\nSaved:")
print(OUT_MANIFEST)

# ------------------------------------------------------------
# 11. Save locked evaluation configuration
# ------------------------------------------------------------

config = {
    "experiment": "beach_supervised_unet_resnet50_wwtp_external",

    "task": "binary_semantic_segmentation",

    "training_domain": "Beach",

    "training_split": {
        "train": 790,
        "validation": 169,
        "primary_test": 169
    },

    "external_domain": "WWTP",

    "wwtp_total_images": 2010,

    "wwtp_original_split_metadata": {
        "train": 1406,
        "validation": 302,
        "test": 302
    },

    "wwtp_project_role": (
        "All WWTP images are treated as an unseen external "
        "evaluation population."
    ),

    "wwtp_training": False,

    "wwtp_validation": False,

    "wwtp_model_selection": False,

    "wwtp_threshold_selection": False,

    "wwtp_primary_metric_population": 2004,

    "wwtp_annotation_empty_population": 6,

    "annotation_empty_policy": (
        "Annotation-empty images are not treated as confirmed "
        "negative ground truth. They are excluded from primary "
        "quantitative metrics and retained for secondary analysis."
    ),

    "original_wwtp_split_usage": (
        "Original train/validation/test labels are preserved "
        "only as dataset provenance and may be used for "
        "secondary stratified reporting."
    ),

    "threshold": 0.5,

    "image_native_resolution": [
        3840,
        2160
    ],

    "tile_size": [
        512,
        512
    ],

    "tiling_stride": [
        384,
        384
    ],

    "tile_overlap_pixels": 128,

    "tile_overlap_fraction": 0.25,

    "inference_resolution": "native_3840x2160",

    "stitching": "overlap_average_logits",

    "primary_metrics": [
        "dice",
        "iou",
        "precision",
        "recall",
        "pixel_accuracy"
    ]
}

with open(OUT_CONFIG, "w") as f:
    json.dump(config, f, indent=2)

print("Saved:")
print(OUT_CONFIG)

# ------------------------------------------------------------
# 12. Final audit table
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("WWTP EXTERNAL EVALUATION AUDIT")
print("=" * 60)

print(f"Total images              : {len(df)}")
print(
    f"Original train metadata   : "
    f"{(df['split'] == 'train').sum()}"
)
print(
    f"Original validation       : "
    f"{(df['split'] == 'validation').sum()}"
)
print(
    f"Original test metadata    : "
    f"{(df['split'] == 'test').sum()}"
)
print(
    f"Primary metric eligible   : "
    f"{df['primary_metric_eligible'].sum()}"
)
print(
    f"Annotation-empty          : "
    f"{df['secondary_analysis_eligible'].sum()}"
)
print(
    f"Used for training        : "
    f"{df['external_evaluation'].eq(False).sum()}"
)
print(
    f"Used for validation      : "
    f"{(
        df['project_split_role']
        .isin(['validation', 'model_selection'])
    ).sum()}"
)

print("=" * 60)

print("\nPASS: WWTP is locked as an external evaluation domain.")

#Step 8: initialize the pretrained U-Net + ResNet-50 model.

In [ ]:
# ============================================================
# STEP 8 — INITIALIZE PRETRAINED U-NET + RESNET-50
# ============================================================

!pip install -q segmentation-models-pytorch

import torch
import segmentation_models_pytorch as smp

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

model = model.to(device)

# ------------------------------------------------------------
# Basic verification
# ------------------------------------------------------------

print("\nModel type:", type(model).__name__)
print("Encoder:", "ResNet-50")
print("Encoder weights:", "ImageNet pretrained")
print("Input channels:", 3)
print("Output classes:", 1)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("\nTotal parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

# ------------------------------------------------------------
# Check encoder weights are loaded
# ------------------------------------------------------------

encoder_params = list(model.encoder.parameters())

print(
    "\nEncoder parameters:",
    len(encoder_params)
)

print(
    "Encoder requires_grad:",
    any(p.requires_grad for p in encoder_params)
)

print("\nSTEP 8 STATUS: MODEL INITIALIZATION COMPLETE")

#Step 9

In [ ]:
# ============================================================
# STEP 9 — FORWARD PASS VERIFICATION
# ============================================================

import torch

# ------------------------------------------------------------
# Get one actual batch from the Beach training DataLoader
# ------------------------------------------------------------

batch = next(iter(train_loader))
images = batch["image"]
masks = batch["mask"]

print("Input batch before device transfer:")
print("Images shape:", images.shape)
print("Masks shape :", masks.shape)
print("Images dtype:", images.dtype)
print("Masks dtype :", masks.dtype)

# ------------------------------------------------------------
# Move batch to GPU
# ------------------------------------------------------------

images = images.to(device)
masks = masks.to(device)

print("\nInput batch after device transfer:")
print("Images shape:", images.shape)
print("Masks shape :", masks.shape)
print("Device:", images.device)

# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

model.eval()

with torch.no_grad():
    logits = model(images)

# ------------------------------------------------------------
# Verify output
# ------------------------------------------------------------

print("\nModel output:")
print("Logits shape:", logits.shape)
print("Logits dtype:", logits.dtype)
print("Logits device:", logits.device)

# ------------------------------------------------------------
# Expected shape verification
# ------------------------------------------------------------

expected_shape = (
    images.shape[0],
    1,
    512,
    512
)

print("\nExpected output shape:", expected_shape)

assert logits.shape == expected_shape, (
    f"Unexpected output shape: {logits.shape}, "
    f"expected {expected_shape}"
)

assert torch.isfinite(logits).all(), (
    "Model output contains NaN or Inf values."
)

print("\nFinite logits: PASS")
print("Output shape: PASS")
print("\nSTEP 9 STATUS: FORWARD PASS VERIFIED")

#Step 10 only: loss + backward-pass verification.

In [ ]:
# ============================================================
# STEP 10 — LOSS + BACKWARD-PASS VERIFICATION
# ============================================================

import torch
import torch.nn as nn

# ------------------------------------------------------------
# 1. Get one real Beach training batch
# ------------------------------------------------------------

batch = next(iter(train_loader))
images = batch["image"].to(device)
masks = batch["mask"].to(device)

# ------------------------------------------------------------
# 2. Dice loss
# ------------------------------------------------------------

def dice_loss(logits, targets, smooth=1.0):
    """
    Binary Dice loss using sigmoid probabilities.
    """

    probs = torch.sigmoid(logits)

    probs = probs.contiguous().view(probs.size(0), -1)
    targets = targets.contiguous().view(targets.size(0), -1)

    intersection = (probs * targets).sum(dim=1)

    dice = (
        (2.0 * intersection + smooth)
        /
        (probs.sum(dim=1) + targets.sum(dim=1) + smooth)
    )

    return 1.0 - dice.mean()


# ------------------------------------------------------------
# 3. Loss functions
# ------------------------------------------------------------

bce_loss_fn = nn.BCEWithLogitsLoss()

# ------------------------------------------------------------
# 4. Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ------------------------------------------------------------
# 5. Set model to training mode
# ------------------------------------------------------------

model.train()

# ------------------------------------------------------------
# 6. Save one parameter before update
# ------------------------------------------------------------

tracked_parameter = next(
    p for p in model.parameters()
    if p.requires_grad
)

parameter_before = tracked_parameter.detach().clone()

# ------------------------------------------------------------
# 7. Forward pass
# ------------------------------------------------------------

optimizer.zero_grad(set_to_none=True)

logits = model(images)

# ------------------------------------------------------------
# 8. Calculate BCE + Dice
# ------------------------------------------------------------

bce = bce_loss_fn(logits, masks)

dice = dice_loss(logits, masks)

total_loss = 0.5 * bce + 0.5 * dice

# ------------------------------------------------------------
# 9. Verify losses
# ------------------------------------------------------------

print("Loss values:")
print("BCE loss :", bce.item())
print("Dice loss:", dice.item())
print("Total loss:", total_loss.item())

assert torch.isfinite(bce), "BCE loss contains NaN or Inf."
assert torch.isfinite(dice), "Dice loss contains NaN or Inf."
assert torch.isfinite(total_loss), "Total loss contains NaN or Inf."

print("\nLoss finite: PASS")

# ------------------------------------------------------------
# 10. Backward pass
# ------------------------------------------------------------

total_loss.backward()

# ------------------------------------------------------------
# 11. Check gradients
# ------------------------------------------------------------

gradient_count = 0
nonzero_gradient_count = 0
finite_gradient_count = 0

for parameter in model.parameters():

    if parameter.requires_grad and parameter.grad is not None:

        gradient_count += 1

        if torch.any(parameter.grad != 0):
            nonzero_gradient_count += 1

        if torch.isfinite(parameter.grad).all():
            finite_gradient_count += 1


print("\nGradient verification:")
print("Parameters with gradients:",
      gradient_count)

print("Parameters with non-zero gradients:",
      nonzero_gradient_count)

print("Parameters with finite gradients:",
      finite_gradient_count)

assert gradient_count > 0, (
    "No gradients were produced."
)

assert nonzero_gradient_count > 0, (
    "All gradients are zero."
)

assert finite_gradient_count == gradient_count, (
    "At least one gradient contains NaN or Inf."
)

print("Gradient computation: PASS")

# ------------------------------------------------------------
# 12. Optimizer update
# ------------------------------------------------------------

optimizer.step()

# ------------------------------------------------------------
# 13. Verify parameter update
# ------------------------------------------------------------

parameter_after = tracked_parameter.detach().clone()

parameter_changed = not torch.equal(
    parameter_before,
    parameter_after
)

print("\nParameter update:")
print("Tracked parameter changed:", parameter_changed)

assert parameter_changed, (
    "Optimizer step did not change the tracked parameter."
)

print("AdamW parameter update: PASS")

# ------------------------------------------------------------
# 14. Final status
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 10 STATUS: LOSS + BACKWARD PASS VERIFIED")
print("=" * 60)

#Step 11 — configure the actual training pipeline

In [ ]:
# ============================================================
# STEP 11 — RESET MODEL + VERIFY TRAINING CONFIGURATION
# ============================================================

import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

# ------------------------------------------------------------
# 1. Reinitialize model from ImageNet pretrained weights
# ------------------------------------------------------------

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(device)

print("Fresh model initialized.")
print("Device:", device)

# ------------------------------------------------------------
# 2. Verify model parameters are trainable
# ------------------------------------------------------------

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("\nModel parameters:")
print("Total parameters     :", f"{total_params:,}")
print("Trainable parameters :", f"{trainable_params:,}")

assert trainable_params == total_params, (
    "Not all model parameters are trainable."
)

print("All model parameters trainable: PASS")

# ------------------------------------------------------------
# 3. Loss functions
# ------------------------------------------------------------

bce_loss_fn = nn.BCEWithLogitsLoss()


def dice_loss(logits, targets, smooth=1.0):

    probs = torch.sigmoid(logits)

    probs = probs.contiguous().view(
        probs.size(0), -1
    )

    targets = targets.contiguous().view(
        targets.size(0), -1
    )

    intersection = (probs * targets).sum(dim=1)

    dice = (
        (2.0 * intersection + smooth)
        /
        (
            probs.sum(dim=1)
            + targets.sum(dim=1)
            + smooth
        )
    )

    return 1.0 - dice.mean()


def combined_loss(logits, targets):

    bce = bce_loss_fn(logits, targets)
    dice = dice_loss(logits, targets)

    total = 0.5 * bce + 0.5 * dice

    return total, bce, dice


# ------------------------------------------------------------
# 4. Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("\nOptimizer:")
print("Type         : AdamW")
print("Learning rate: 1e-4")
print("Weight decay : 1e-4")

# ------------------------------------------------------------
# 5. Training configuration
# ------------------------------------------------------------

NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 10
BATCH_SIZE = 8

print("\nTraining configuration:")
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", NUM_EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)

# ------------------------------------------------------------
# 6. Verify training DataLoader
# ------------------------------------------------------------

train_batch = next(iter(train_loader))
train_batch_images = train_batch["image"]
train_batch_masks = train_batch["mask"]

print("\nTraining DataLoader:")
print("Images:", train_batch_images.shape)
print("Masks :", train_batch_masks.shape)

assert train_batch_images.shape == (
    BATCH_SIZE, 3, 512, 512
)

assert train_batch_masks.shape == (
    BATCH_SIZE, 1, 512, 512
)

print("Training DataLoader shape: PASS")

# ------------------------------------------------------------
# 7. Verify validation DataLoader
# ------------------------------------------------------------

val_batch = next(iter(val_loader))
val_batch_images = val_batch["image"]
val_batch_masks = val_batch["mask"]

print("\nValidation DataLoader:")
print("Images:", val_batch_images.shape)
print("Masks :", val_batch_masks.shape)

assert val_batch_images.shape == (
    BATCH_SIZE, 3, 512, 512
)

assert val_batch_masks.shape == (
    BATCH_SIZE, 1, 512, 512
)

print("Validation DataLoader shape: PASS")

# ------------------------------------------------------------
# 8. Verify test DataLoader exists
# ------------------------------------------------------------

test_batch = next(iter(test_loader))
test_batch_images = test_batch["image"]
test_batch_masks = test_batch["mask"]

print("\nBeach test DataLoader:")
print("Images:", test_batch_images.shape)
print("Masks :", test_batch_masks.shape)

assert test_batch_images.shape == (
    BATCH_SIZE, 3, 512, 512
)

assert test_batch_masks.shape == (
    BATCH_SIZE, 1, 512, 512
)

print("Beach test DataLoader shape: PASS")

# ------------------------------------------------------------
# 9. Verify no WWTP loader is involved
# ------------------------------------------------------------

print("\nWWTP:")
print("Training:     NO")
print("Validation:   NO")
print("Model select: NO")
print("External eval: YES")

# ------------------------------------------------------------
# 10. Final status
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 11 STATUS: TRAINING CONFIGURATION VERIFIED")
print("=" * 60)

#Step 12: actual training.

In [ ]:
# ============================================================
# STEP 12 — BASELINE TRAINING
# Beach Train → Beach Validation
# ============================================================

import os
import time
import torch

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

NUM_EPOCHS = 50
PATIENCE = 10

CHECKPOINT_DIR = os.path.join(
    "/content/drive/MyDrive/MicroPlastics/research_workspace",
    "analysis",
    "baseline_experiment",
    "checkpoints"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,
    "beach_unet_resnet50_best_dice.pth"
)

# ------------------------------------------------------------
# Dice metric
# ------------------------------------------------------------

def dice_score(logits, targets, threshold=0.5, smooth=1.0):

    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    preds = preds.contiguous().view(preds.size(0), -1)
    targets = targets.contiguous().view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)

    dice = (
        (2.0 * intersection + smooth)
        /
        (
            preds.sum(dim=1)
            + targets.sum(dim=1)
            + smooth
        )
    )

    return dice.mean().item()


# ------------------------------------------------------------
# IoU metric
# ------------------------------------------------------------

def iou_score(logits, targets, threshold=0.5, smooth=1.0):

    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    preds = preds.contiguous().view(preds.size(0), -1)
    targets = targets.contiguous().view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)

    union = (
        preds.sum(dim=1)
        + targets.sum(dim=1)
        - intersection
    )

    iou = (
        (intersection + smooth)
        /
        (union + smooth)
    )

    return iou.mean().item()


# ------------------------------------------------------------
# Training history
# ------------------------------------------------------------

history = []

best_val_dice = -1.0
epochs_without_improvement = 0

# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

for epoch in range(1, NUM_EPOCHS + 1):

    epoch_start = time.time()

    # ========================================================
    # TRAINING
    # ========================================================

    model.train()

    train_loss_sum = 0.0
    train_batches = 0

    for batch in train_loader:

        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)

        loss, bce, dice = combined_loss(
            logits,
            masks
        )

        loss.backward()

        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    train_loss = train_loss_sum / train_batches

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss_sum = 0.0
    val_dice_sum = 0.0
    val_iou_sum = 0.0
    val_batches = 0

    with torch.no_grad():

        for batch in val_loader:

            images = batch["image"].to(device, non_blocking=True)
            masks = batch["mask"].to(device, non_blocking=True)

            logits = model(images)

            loss, bce, dice = combined_loss(
                logits,
                masks
            )

            batch_dice = dice_score(
                logits,
                masks,
                threshold=0.5
            )

            batch_iou = iou_score(
                logits,
                masks,
                threshold=0.5
            )

            val_loss_sum += loss.item()
            val_dice_sum += batch_dice
            val_iou_sum += batch_iou
            val_batches += 1

    val_loss = val_loss_sum / val_batches
    val_dice = val_dice_sum / val_batches
    val_iou = val_iou_sum / val_batches

    epoch_time = time.time() - epoch_start

    # ========================================================
    # SAVE HISTORY
    # ========================================================

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_dice": val_dice,
        "val_iou": val_iou,
        "epoch_time_sec": epoch_time
    }

    history.append(epoch_record)

    # ========================================================
    # CHECKPOINT
    # ========================================================

    improved = val_dice > best_val_dice

    if improved:

        best_val_dice = val_dice
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_dice": best_val_dice,
                "val_iou": val_iou,
                "train_loss": train_loss,
                "val_loss": val_loss
            },
            BEST_MODEL_PATH
        )

        checkpoint_status = "BEST CHECKPOINT SAVED"

    else:

        epochs_without_improvement += 1
        checkpoint_status = (
            f"no improvement "
            f"({epochs_without_improvement}/{PATIENCE})"
        )

    # ========================================================
    # PRINT EPOCH RESULTS
    # ========================================================

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"Val IoU: {val_iou:.4f} | "
        f"{checkpoint_status}"
    )

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping triggered at epoch {epoch}."
        )

        break


# ------------------------------------------------------------
# Final training summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 12 TRAINING COMPLETE")
print("=" * 70)

print(
    "Best validation Dice:",
    f"{best_val_dice:.4f}"
)

print(
    "Best checkpoint:",
    BEST_MODEL_PATH
)

print(
    "Epochs completed:",
    len(history)
)

However, there is a genuine issue in the logging: Step 12 only printed train_loss, not the separate BCE and Dice values. Therefore, from the epoch output alone, you cannot see how much BCE contributed.

In [ ]:
# ============================================================
# STEP 12B — SAVE COMPLETE BASELINE TRAINING ARTIFACT
# ============================================================

import os
import json
import torch

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASELINE_DIR = (
    "/content/drive/MyDrive/MicroPlastics/"
    "research_workspace/analysis/baseline_experiment"
)

CHECKPOINT_DIR = os.path.join(
    BASELINE_DIR,
    "checkpoints"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

FULL_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "beach_unet_resnet50_baseline_full_checkpoint.pth"
)

HISTORY_PATH = os.path.join(
    BASELINE_DIR,
    "beach_baseline_training_history.json"
)

CONFIG_PATH = os.path.join(
    BASELINE_DIR,
    "beach_baseline_training_config.json"
)

# ------------------------------------------------------------
# 1. Save complete training checkpoint
# ------------------------------------------------------------

full_checkpoint = {
    "experiment_name":
        "beach_supervised_unet_resnet50_baseline",

    "model_architecture": {
        "architecture": "U-Net",
        "encoder": "ResNet-50",
        "encoder_weights": "ImageNet",
        "in_channels": 3,
        "classes": 1
    },

    "model_state_dict":
        model.state_dict(),

    "optimizer_state_dict":
        optimizer.state_dict(),

    "best_val_dice":
        best_val_dice,

    "completed_epochs":
        len(history),

    "history":
        history,

    "loss": {
        "bce_weight": 0.5,
        "dice_weight": 0.5,
        "description":
            "0.5 BCEWithLogitsLoss + 0.5 Dice Loss"
    },

    "optimizer": {
        "type": "AdamW",
        "learning_rate": 1e-4,
        "weight_decay": 1e-4
    },

    "training": {
        "batch_size": 8,
        "max_epochs": 50,
        "early_stopping_patience": 10,
        "checkpoint_metric": "validation_dice",
        "threshold": 0.5
    },

    "data": {
        "training_domain": "Beach",
        "train_images": 790,
        "validation_images": 169,
        "primary_test_images": 169,
        "wwtp_training": False,
        "wwtp_validation": False,
        "wwtp_external_evaluation": True
    },

    "device": str(device)
}

torch.save(
    full_checkpoint,
    FULL_CHECKPOINT_PATH
)

print("Full checkpoint saved:")
print(FULL_CHECKPOINT_PATH)

# ------------------------------------------------------------
# 2. Save training history separately
# ------------------------------------------------------------

with open(HISTORY_PATH, "w") as f:
    json.dump(history, f, indent=2)

print("\nTraining history saved:")
print(HISTORY_PATH)

# ------------------------------------------------------------
# 3. Save human-readable configuration
# ------------------------------------------------------------

training_config = {
    "experiment_name":
        "beach_supervised_unet_resnet50_baseline",

    "architecture":
        "U-Net with ResNet-50 encoder",

    "encoder_pretraining":
        "ImageNet",

    "input_shape":
        [3, 512, 512],

    "output_shape":
        [1, 512, 512],

    "loss":
        "0.5 BCEWithLogitsLoss + 0.5 Dice Loss",

    "optimizer":
        "AdamW",

    "learning_rate":
        1e-4,

    "weight_decay":
        1e-4,

    "batch_size":
        8,

    "maximum_epochs":
        50,

    "early_stopping_patience":
        10,

    "checkpoint_selection":
        "Best validation Dice",

    "best_epoch":
        history[
            max(
                range(len(history)),
                key=lambda i: history[i]["val_dice"]
            )
        ]["epoch"],

    "best_validation_dice":
        best_val_dice,

    "best_checkpoint":
        BEST_MODEL_PATH,

    "full_checkpoint":
        FULL_CHECKPOINT_PATH,

    "beach_train":
        790,

    "beach_validation":
        169,

    "beach_primary_test":
        169,

    "wwtp_used_for_training":
        False,

    "wwtp_used_for_validation":
        False,

    "wwtp_role":
        "External evaluation only"
}

with open(CONFIG_PATH, "w") as f:
    json.dump(training_config, f, indent=2)

print("\nTraining configuration saved:")
print(CONFIG_PATH)

# ------------------------------------------------------------
# 4. Verify files exist
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CHECKPOINT STORAGE VERIFICATION")
print("=" * 60)

for path in [
    BEST_MODEL_PATH,
    FULL_CHECKPOINT_PATH,
    HISTORY_PATH,
    CONFIG_PATH
]:

    exists = os.path.exists(path)

    size_mb = (
        os.path.getsize(path) / (1024 ** 2)
        if exists else 0
    )

    print(
        f"{os.path.basename(path):55s}"
        f" | exists={exists}"
        f" | size={size_mb:.2f} MB"
    )

    assert exists, f"File was not saved: {path}"

print("=" * 60)
print("STEP 12B STATUS: TRAINING ARTIFACTS SAFELY SAVED")
print("=" * 60)

#Step 13 — Beach test evaluation

In [ ]:
# ============================================================
# STEP 13 — BEACH TEST EVALUATION
# ============================================================

import os
import torch

# ------------------------------------------------------------
# 1. Load BEST checkpoint
# ------------------------------------------------------------

BEST_MODEL_PATH = (
    "/content/drive/MyDrive/MicroPlastics/"
    "research_workspace/analysis/baseline_experiment/"
    "checkpoints/beach_unet_resnet50_best_dice.pth"
)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

print("Loaded checkpoint:")
print(BEST_MODEL_PATH)

print(
    "\nCheckpoint epoch:",
    checkpoint["epoch"]
)

print(
    "Best validation Dice:",
    checkpoint["best_val_dice"]
)

# ------------------------------------------------------------
# 2. Metric functions
# ------------------------------------------------------------

def segmentation_metrics(
    logits,
    targets,
    threshold=0.5,
    smooth=1e-6
):

    probs = torch.sigmoid(logits)

    preds = (probs >= threshold).float()

    # Flatten per image
    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    # --------------------------------------------------------
    # TP / FP / FN / TN
    # --------------------------------------------------------

    tp = (preds * targets).sum(dim=1)

    fp = (preds * (1 - targets)).sum(dim=1)

    fn = ((1 - preds) * targets).sum(dim=1)

    tn = ((1 - preds) * (1 - targets)).sum(dim=1)

    # --------------------------------------------------------
    # Dice
    # --------------------------------------------------------

    dice = (
        (2 * tp + smooth)
        /
        (2 * tp + fp + fn + smooth)
    )

    # --------------------------------------------------------
    # IoU
    # --------------------------------------------------------

    iou = (
        (tp + smooth)
        /
        (tp + fp + fn + smooth)
    )

    # --------------------------------------------------------
    # Precision
    # --------------------------------------------------------

    precision = (
        (tp + smooth)
        /
        (tp + fp + smooth)
    )

    # --------------------------------------------------------
    # Recall
    # --------------------------------------------------------

    recall = (
        (tp + smooth)
        /
        (tp + fn + smooth)
    )

    # --------------------------------------------------------
    # Pixel accuracy
    # --------------------------------------------------------

    accuracy = (
        (tp + tn + smooth)
        /
        (tp + tn + fp + fn + smooth)
    )

    return {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "pixel_accuracy": accuracy
    }


# ------------------------------------------------------------
# 3. Evaluate Beach test set
# ------------------------------------------------------------

test_results = {
    "dice": [],
    "iou": [],
    "precision": [],
    "recall": [],
    "pixel_accuracy": []
}

total_images = 0

with torch.no_grad():

    for batch in test_loader:

        images = batch["image"].to(
            device,
            non_blocking=True
)

        masks = batch["mask"].to(
            device,
            non_blocking=True
        )

        logits = model(images)

        metrics = segmentation_metrics(
            logits,
            masks,
            threshold=0.5
        )

        batch_size = images.size(0)

        total_images += batch_size

        for key in test_results:

            test_results[key].extend(
                metrics[key]
                .detach()
                .cpu()
                .tolist()
            )


# ------------------------------------------------------------
# 4. Calculate final test metrics
# ------------------------------------------------------------

test_summary = {}

for key, values in test_results.items():

    test_summary[key] = sum(values) / len(values)


# ------------------------------------------------------------
# 5. Print results
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BEACH TEST RESULTS")
print("=" * 65)

print("Images evaluated:", total_images)

print(
    f"Dice          : {test_summary['dice']:.4f}"
)

print(
    f"IoU           : {test_summary['iou']:.4f}"
)

print(
    f"Precision     : {test_summary['precision']:.4f}"
)

print(
    f"Recall        : {test_summary['recall']:.4f}"
)

print(
    f"Pixel Accuracy: {test_summary['pixel_accuracy']:.4f}"
)

print("=" * 65)

# ------------------------------------------------------------
# 6. Verify expected test population
# ------------------------------------------------------------

assert total_images == 169, (
    f"Expected 169 test images, got {total_images}"
)

# Verify all metrics are finite

for key, value in test_summary.items():

    assert torch.isfinite(
        torch.tensor(value)
    ), f"{key} is not finite"

print("\nTest population: PASS")
print("All metrics finite: PASS")

print(
    "\nSTEP 13 STATUS: BEACH TEST EVALUATION COMPLETE"
)

#Step 14 is the important cross-domain experiment

In [ ]:
# ============================================================
# STEP 14A — WWTP TILING + STITCHING VERIFICATION
# One real annotated-positive WWTP image
# ============================================================

import os
import math
import numpy as np
from PIL import Image
import torch

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TILE_SIZE = 512
STRIDE = 384
THRESHOLD = 0.5

WWTP_MANIFEST = (
    "/content/drive/MyDrive/MicroPlastics/"
    "research_workspace/analysis/baseline_experiment/"
    "wwtp_external_evaluation_manifest.csv"
)

# ------------------------------------------------------------
# 1. Load external WWTP manifest
# ------------------------------------------------------------

wwtp_df = pd.read_csv(WWTP_MANIFEST)

# Select one primary-metric-eligible image
sample_row = wwtp_df[
    wwtp_df["primary_metric_eligible"] == True
].iloc[0]

image_path = sample_row["image_path"]
mask_path = sample_row["mask_path"]

print("Selected WWTP sample:")
print("sample_id:", sample_row["sample_id"])
print("original split:", sample_row["split"])
print("image:", image_path)
print("mask :", mask_path)

# ------------------------------------------------------------
# 2. Load image and ground-truth mask
# ------------------------------------------------------------

image = Image.open(image_path).convert("RGB")
gt_mask = Image.open(mask_path).convert("L")

image_np = np.asarray(image)
gt_np = np.asarray(gt_mask)

print("\nNative image shape:", image_np.shape)
print("Native mask shape :", gt_np.shape)

assert image_np.shape[:2] == gt_np.shape[:2]

height, width = image_np.shape[:2]

assert (width, height) == (3840, 2160), (
    f"Unexpected WWTP resolution: {(width, height)}"
)

print("Native resolution check: PASS")

# ------------------------------------------------------------
# 3. Generate tile start positions
# ------------------------------------------------------------

def get_positions(length, tile_size, stride):

    positions = list(
        range(
            0,
            max(length - tile_size, 0) + 1,
            stride
        )
    )

    final_position = length - tile_size

    if positions[-1] != final_position:
        positions.append(final_position)

    return sorted(set(positions))


x_positions = get_positions(
    width,
    TILE_SIZE,
    STRIDE
)

y_positions = get_positions(
    height,
    TILE_SIZE,
    STRIDE
)

print("\nTile positions:")
print("X positions:", x_positions)
print("Y positions:", y_positions)

print(
    "\nNumber of X positions:",
    len(x_positions)
)

print(
    "Number of Y positions:",
    len(y_positions)
)

expected_tile_count = (
    len(x_positions) * len(y_positions)
)

print(
    "Total tiles:",
    expected_tile_count
)

# ------------------------------------------------------------
# 4. Verify coverage
# ------------------------------------------------------------

coverage = np.zeros(
    (height, width),
    dtype=np.uint16
)

for y in y_positions:

    for x in x_positions:

        coverage[
            y:y + TILE_SIZE,
            x:x + TILE_SIZE
        ] += 1

print("\nCoverage:")
print("Minimum coverage:", coverage.min())
print("Maximum coverage:", coverage.max())

assert coverage.min() >= 1, (
    "Some native pixels are not covered by any tile."
)

print("Full native-image coverage: PASS")

# ------------------------------------------------------------
# 5. Load frozen best Beach model
# ------------------------------------------------------------

BEST_MODEL_PATH = (
    "/content/drive/MyDrive/MicroPlastics/"
    "research_workspace/analysis/baseline_experiment/"
    "checkpoints/beach_unet_resnet50_best_dice.pth"
)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

# Explicitly freeze parameters
for parameter in model.parameters():
    parameter.requires_grad = False

print("\nModel:")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation Dice:", checkpoint["best_val_dice"])
print("Model frozen: PASS")

assert not any(
    p.requires_grad for p in model.parameters()
)

# ------------------------------------------------------------
# 6. Prepare image normalization
# ------------------------------------------------------------

# IMPORTANT:
# The current Beach preprocessing pipeline verified earlier
# produces float32 RGB tensors in the range [0,1].
#
# We therefore use the same basic conversion here.

image_float = (
    image_np.astype(np.float32) / 255.0
)

# ------------------------------------------------------------
# 7. Tile inference
# ------------------------------------------------------------

logit_sum = np.zeros(
    (height, width),
    dtype=np.float32
)

logit_count = np.zeros(
    (height, width),
    dtype=np.float32
)

tile_counter = 0

with torch.no_grad():

    for y in y_positions:

        for x in x_positions:

            tile = image_float[
                y:y + TILE_SIZE,
                x:x + TILE_SIZE,
                :
            ]

            # HWC → CHW
            tile_tensor = torch.from_numpy(
                tile.transpose(2, 0, 1)
            ).float()

            # Add batch dimension
            tile_tensor = tile_tensor.unsqueeze(0)

            tile_tensor = tile_tensor.to(device)

            # Model prediction
            tile_logits = model(
                tile_tensor
            )

            assert tile_logits.shape == (
                1, 1, TILE_SIZE, TILE_SIZE
            )

            tile_logits_np = (
                tile_logits[0, 0]
                .detach()
                .cpu()
                .numpy()
            )

            # Accumulate logits
            logit_sum[
                y:y + TILE_SIZE,
                x:x + TILE_SIZE
            ] += tile_logits_np

            logit_count[
                y:y + TILE_SIZE,
                x:x + TILE_SIZE
            ] += 1

            tile_counter += 1

# ------------------------------------------------------------
# 8. Stitch overlapping logits
# ------------------------------------------------------------

assert tile_counter == expected_tile_count

assert np.all(logit_count > 0), (
    "Some pixels have no tile prediction."
)

stitched_logits = (
    logit_sum / logit_count
)

print("\nStitching:")
print("Tiles processed:", tile_counter)
print("Expected tiles :", expected_tile_count)
print(
    "Stitched shape  :",
    stitched_logits.shape
)

assert stitched_logits.shape == (
    height,
    width
)

print("Stitched resolution: PASS")

# ------------------------------------------------------------
# 9. Convert logits → probability
# ------------------------------------------------------------

probability_map = 1.0 / (
    1.0 + np.exp(
        -np.clip(stitched_logits, -50, 50)
    )
)

print("\nProbability map:")
print("Min:", probability_map.min())
print("Max:", probability_map.max())
print("Mean:", probability_map.mean())

assert np.isfinite(
    probability_map
).all()

assert probability_map.min() >= 0
assert probability_map.max() <= 1

print("Probability range: PASS")

# ------------------------------------------------------------
# 10. Threshold
# ------------------------------------------------------------

prediction_mask = (
    probability_map >= THRESHOLD
).astype(np.uint8)

print("\nPrediction mask:")
print("Shape:", prediction_mask.shape)
print("Unique values:", np.unique(prediction_mask))

assert prediction_mask.shape == (
    height,
    width
)

assert set(
    np.unique(prediction_mask)
).issubset({0, 1})

print("Binary mask generation: PASS")

# ------------------------------------------------------------
# 11. Compare with native ground truth dimensions
# ------------------------------------------------------------

gt_binary = (
    gt_np > 0
).astype(np.uint8)

print("\nGround truth:")
print("Shape:", gt_binary.shape)
print("Unique values:", np.unique(gt_binary))

assert gt_binary.shape == prediction_mask.shape

print("Prediction / GT spatial alignment: PASS")

# ------------------------------------------------------------
# 12. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 14A STATUS: WWTP TILE + STITCH VERIFICATION COMPLETE")
print("=" * 70)

print("Native image       :", (width, height))
print("Tile size          :", TILE_SIZE)
print("Stride             :", STRIDE)
print("Overlap            :", TILE_SIZE - STRIDE, "pixels")
print("Total tiles        :", tile_counter)
print("Stitched prediction:", stitched_logits.shape)
print("Threshold          :", THRESHOLD)
print("Model frozen       : YES")
print("=" * 70)

#Step 14B — Full WWTP external evaluation

The evaluation will:
1. Use the epoch-48 best Beach checkpoint.
2. Keep the model frozen.
3. Process all 2,010 WWTP images using the verified 512/384 tiling.
4. Calculate primary metrics only on the 2,004 annotated_positive images.
5. Keep the 6 annotation_empty images out of primary metrics.
6. Preserve the original WWTP train/validation/test labels so we can report a secondary breakdown if useful.
7. Save the per-image results to Drive so the expensive inference does not need to be repeated.

In [ ]:
# ============================================================
# STEP 14B — FULL WWTP EXTERNAL EVALUATION
# ============================================================

import os
import json
import time
import numpy as np
import pandas as pd
from PIL import Image
import torch

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TILE_SIZE = 512
STRIDE = 384
THRESHOLD = 0.5

WWTP_MANIFEST = (
    "/content/drive/MyDrive/MicroPlastics/"
    "research_workspace/analysis/baseline_experiment/"
    "wwtp_external_evaluation_manifest.csv"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/MicroPlastics/"
    "research_workspace/analysis/baseline_experiment/"
    "wwtp_results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "wwtp_external_per_image_results.csv"
)

SUMMARY_PATH = os.path.join(
    RESULTS_DIR,
    "wwtp_external_summary.json"
)

BEST_MODEL_PATH = (
    "/content/drive/MyDrive/MicroPlastics/"
    "research_workspace/analysis/baseline_experiment/"
    "checkpoints/"
    "beach_unet_resnet50_best_dice.pth"
)

# ------------------------------------------------------------
# 1. Load manifest
# ------------------------------------------------------------

wwtp_df = pd.read_csv(WWTP_MANIFEST)

print("WWTP population:", len(wwtp_df))

assert len(wwtp_df) == 2010

# ------------------------------------------------------------
# 2. Load BEST Beach checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

# Freeze model explicitly
for parameter in model.parameters():
    parameter.requires_grad = False

assert not any(
    p.requires_grad for p in model.parameters()
)

print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation Dice:", checkpoint["best_val_dice"])
print("Model frozen: PASS")

# ------------------------------------------------------------
# 3. Tile position function
# ------------------------------------------------------------

def get_positions(length, tile_size, stride):

    positions = list(
        range(
            0,
            max(length - tile_size, 0) + 1,
            stride
        )
    )

    final_position = length - tile_size

    if positions[-1] != final_position:
        positions.append(final_position)

    return sorted(set(positions))


# ------------------------------------------------------------
# 4. Metric function
# ------------------------------------------------------------

def calculate_metrics(pred, target):

    pred = pred.astype(np.uint8)
    target = target.astype(np.uint8)

    pred_bool = pred.astype(bool)
    target_bool = target.astype(bool)

    tp = np.logical_and(
        pred_bool,
        target_bool
    ).sum()

    fp = np.logical_and(
        pred_bool,
        ~target_bool
    ).sum()

    fn = np.logical_and(
        ~pred_bool,
        target_bool
    ).sum()

    tn = np.logical_and(
        ~pred_bool,
        ~target_bool
    ).sum()

    dice = (
        (2.0 * tp) /
        (2.0 * tp + fp + fn)
        if (2.0 * tp + fp + fn) > 0
        else 1.0
    )

    iou = (
        tp /
        (tp + fp + fn)
        if (tp + fp + fn) > 0
        else 1.0
    )

    precision = (
        tp /
        (tp + fp)
        if (tp + fp) > 0
        else 1.0
    )

    recall = (
        tp /
        (tp + fn)
        if (tp + fn) > 0
        else 1.0
    )

    pixel_accuracy = (
        (tp + tn) /
        (tp + tn + fp + fn)
    )

    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "pixel_accuracy": float(pixel_accuracy),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn)
    }


# ------------------------------------------------------------
# 5. Resume support
# ------------------------------------------------------------

completed = {}

if os.path.exists(RESULTS_PATH):

    existing_results = pd.read_csv(
        RESULTS_PATH
    )

    if "sample_id" in existing_results.columns:

        completed = set(
            existing_results["sample_id"]
            .astype(str)
        )

        print(
            "Existing completed samples:",
            len(completed)
        )

else:

    existing_results = pd.DataFrame()


# ------------------------------------------------------------
# 6. Evaluation loop
# ------------------------------------------------------------

results = []

start_time = time.time()

total = len(wwtp_df)

for index, row in wwtp_df.iterrows():

    sample_id = str(row["sample_id"])

    # Resume if already completed
    if sample_id in completed:
        continue

    image_path = row["image_path"]
    mask_path = row["mask_path"]

    # --------------------------------------------------------
    # Load image
    # --------------------------------------------------------

    image = Image.open(
        image_path
    ).convert("RGB")

    image_np = np.asarray(image)

    height, width = image_np.shape[:2]

    assert (
        width == 3840 and height == 2160
    ), (
        f"Unexpected resolution for {sample_id}: "
        f"{width}x{height}"
    )

    # --------------------------------------------------------
    # Load GT
    # --------------------------------------------------------

    gt_mask = Image.open(
        mask_path
    ).convert("L")

    gt_np = np.asarray(gt_mask)

    assert gt_np.shape == (
        height,
        width
    )

    gt_binary = (
        gt_np > 0
    ).astype(np.uint8)

    # --------------------------------------------------------
    # Tile positions
    # --------------------------------------------------------

    x_positions = get_positions(
        width,
        TILE_SIZE,
        STRIDE
    )

    y_positions = get_positions(
        height,
        TILE_SIZE,
        STRIDE
    )

    # --------------------------------------------------------
    # Accumulation arrays
    # --------------------------------------------------------

    logit_sum = np.zeros(
        (height, width),
        dtype=np.float32
    )

    logit_count = np.zeros(
        (height, width),
        dtype=np.float32
    )

    image_float = (
        image_np.astype(np.float32)
        / 255.0
    )

    # --------------------------------------------------------
    # Tiled inference
    # --------------------------------------------------------

    with torch.no_grad():

        for y in y_positions:

            for x in x_positions:

                tile = image_float[
                    y:y + TILE_SIZE,
                    x:x + TILE_SIZE,
                    :
                ]

                tile_tensor = torch.from_numpy(
                    tile.transpose(2, 0, 1)
                ).float()

                tile_tensor = (
                    tile_tensor
                    .unsqueeze(0)
                    .to(device)
                )

                tile_logits = model(
                    tile_tensor
                )

                tile_logits_np = (
                    tile_logits[0, 0]
                    .detach()
                    .cpu()
                    .numpy()
                )

                logit_sum[
                    y:y + TILE_SIZE,
                    x:x + TILE_SIZE
                ] += tile_logits_np

                logit_count[
                    y:y + TILE_SIZE,
                    x:x + TILE_SIZE
                ] += 1

    # --------------------------------------------------------
    # Stitch
    # --------------------------------------------------------

    assert np.all(
        logit_count > 0
    )

    stitched_logits = (
        logit_sum / logit_count
    )

    # --------------------------------------------------------
    # Probability
    # --------------------------------------------------------

    probability = 1.0 / (
        1.0 +
        np.exp(
            -np.clip(
                stitched_logits,
                -50,
                50
            )
        )
    )

    # --------------------------------------------------------
    # Threshold
    # --------------------------------------------------------

    prediction = (
        probability >= THRESHOLD
    ).astype(np.uint8)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    metrics = calculate_metrics(
        prediction,
        gt_binary
    )

    # --------------------------------------------------------
    # Record
    # --------------------------------------------------------

    result = {
        "sample_id": sample_id,
        "original_split": row["split"],
        "mask_status": row["mask_status"],
        "primary_metric_eligible":
            bool(row["primary_metric_eligible"]),

        "height": height,
        "width": width,

        "tile_size": TILE_SIZE,
        "stride": STRIDE,
        "overlap_pixels":
            TILE_SIZE - STRIDE,

        "threshold": THRESHOLD,

        "dice": metrics["dice"],
        "iou": metrics["iou"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "pixel_accuracy":
            metrics["pixel_accuracy"],

        "tp": metrics["tp"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "tn": metrics["tn"]
    }

    results.append(result)

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    processed = (
        len(completed)
        + len(results)
    )

    if (
        processed % 25 == 0
        or processed == total
    ):

        elapsed = (
            time.time() - start_time
        )

        print(
            f"[{processed}/{total}] "
            f"{sample_id} | "
            f"Dice={metrics['dice']:.4f} | "
            f"IoU={metrics['iou']:.4f} | "
            f"Elapsed={elapsed/60:.1f} min"
        )

        # Save intermediate results
        if results:

            new_df = pd.DataFrame(
                results
            )

            if not existing_results.empty:

                save_df = pd.concat(
                    [
                        existing_results,
                        new_df
                    ],
                    ignore_index=True
                )

            else:

                save_df = new_df

            save_df.to_csv(
                RESULTS_PATH,
                index=False
            )

# ------------------------------------------------------------
# 7. Reload complete result file
# ------------------------------------------------------------

all_results = pd.read_csv(
    RESULTS_PATH
)

print("\nTotal completed results:",
      len(all_results))

assert len(all_results) == 2010

# ------------------------------------------------------------
# 8. Primary evaluation population
# ------------------------------------------------------------

primary = all_results[
    all_results["primary_metric_eligible"] == True
].copy()

secondary_empty = all_results[
    all_results["mask_status"] ==
    "annotation_empty"
].copy()

print(
    "Primary metric population:",
    len(primary)
)

print(
    "Annotation-empty population:",
    len(secondary_empty)
)

assert len(primary) == 2004
assert len(secondary_empty) == 6

# ------------------------------------------------------------
# 9. Aggregate primary metrics
# ------------------------------------------------------------

metric_names = [
    "dice",
    "iou",
    "precision",
    "recall",
    "pixel_accuracy"
]

summary = {
    "total_wwtp_images": 2010,
    "primary_metric_images": 2004,
    "annotation_empty_images": 6,

    "checkpoint_epoch":
        int(checkpoint["epoch"]),

    "checkpoint_validation_dice":
        float(checkpoint["best_val_dice"]),

    "tile_size": TILE_SIZE,
    "stride": STRIDE,
    "overlap_pixels":
        TILE_SIZE - STRIDE,
    "threshold": THRESHOLD
}

for metric in metric_names:

    summary[
        f"{metric}_mean"
    ] = float(
        primary[metric].mean()
    )

    summary[
        f"{metric}_std"
    ] = float(
        primary[metric].std()
    )

    summary[
        f"{metric}_median"
    ] = float(
        primary[metric].median()
    )

# ------------------------------------------------------------
# 10. Save summary
# ------------------------------------------------------------

with open(
    SUMMARY_PATH,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 11. Print primary results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WWTP EXTERNAL EVALUATION — PRIMARY RESULTS")
print("=" * 70)

print(
    "Images evaluated:",
    len(primary)
)

for metric in metric_names:

    print(
        f"{metric:15s}: "
        f"{summary[metric + '_mean']:.4f} "
        f"± "
        f"{summary[metric + '_std']:.4f}"
    )

print("\nMedian values:")

for metric in metric_names:

    print(
        f"{metric:15s}: "
        f"{summary[metric + '_median']:.4f}"
    )

print("=" * 70)

print(
    "\nSaved per-image results:",
    RESULTS_PATH
)

print(
    "Saved summary:",
    SUMMARY_PATH
)

print(
    "\nSTEP 14B STATUS: "
    "WWTP EXTERNAL EVALUATION COMPLETE"
)

This gives us a very important experimental finding: the model performs substantially differently when moved from the Beach domain to WWTP imagery. The Beach test performance was Dice 0.8323, whereas the external WWTP Dice is 0.0604. That establishes a large cross-domain performance gap for this baseline.
The pattern is also informative. Recall is considerably higher than precision, while Dice/IoU remain low. In practical terms, the model is detecting foreground-like regions in many WWTP images, but the predicted regions have poor spatial agreement with the WWTP ground-truth masks and contain substantial false-positive area. We should describe this as an observed metric pattern, not speculate about its cause yet.
The result is therefore exactly useful for your research question: it demonstrates that good in-domain Beach segmentation performance does not automatically translate into strong WWTP-domain segmentation.
Your saved artifacts are also in place:
- Per-image results: wwtp_external_per_image_results.csv
- Summary: wwtp_external_summary.json
Both are under your baseline_experiment/wwtp_results/ directory.

In [ ]:
# ============================================================
# STEP 15A — WWTP QUALITATIVE BASELINE ANALYSIS
# ============================================================

import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

RESULTS_CSV = (
    "/content/drive/MyDrive/MicroPlastics/research_workspace/"
    "analysis/baseline_experiment/wwtp_results/"
    "wwtp_external_per_image_results.csv"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/MicroPlastics/research_workspace/"
    "analysis/baseline_experiment/step15_analysis"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Load per-image results
# ------------------------------------------------------------

df = pd.read_csv(RESULTS_CSV)

print("WWTP result rows:", len(df))
print("Columns:")
print(df.columns.tolist())

# ------------------------------------------------------------
# Inspect available path columns
# ------------------------------------------------------------

print("\nFirst row:")
print(df.iloc[0].to_dict())

# ------------------------------------------------------------
# Identify useful columns
# ------------------------------------------------------------

required_metrics = ["dice", "iou"]

for col in required_metrics:
    assert col in df.columns, f"Missing required column: {col}"

# ------------------------------------------------------------
# Select representative samples
#
# 1. Lowest Dice
# 2. Low Dice
# 3. Median-ish Dice
# 4. High Dice
# 5. Highest Dice
# ------------------------------------------------------------

df_sorted = df.sort_values("dice").reset_index(drop=True)

n = len(df_sorted)

candidate_indices = [
    0,
    int(0.10 * (n - 1)),
    int(0.25 * (n - 1)),
    int(0.50 * (n - 1)),
    int(0.75 * (n - 1)),
    int(0.90 * (n - 1)),
    n - 1,
]

selected = df_sorted.iloc[candidate_indices].copy()

# Remove accidental duplicates
selected = selected.drop_duplicates(subset=["sample_id"])

print("\nSelected representative samples:")
print(
    selected[
        [c for c in ["sample_id", "dice", "iou", "precision", "recall"]
         if c in selected.columns]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# Find image/mask paths
# ------------------------------------------------------------

def find_path(row, possible_columns):
    for col in possible_columns:
        if col in row.index:
            value = row[col]
            if isinstance(value, str) and os.path.exists(value):
                return value
    return None


# ------------------------------------------------------------
# IMPORTANT:
# The existing evaluation CSV should contain image/mask paths.
# If not, reconstruct them from sample_id.
# ------------------------------------------------------------

def reconstruct_image_path(sample_id):
    if sample_id.startswith("wwtp_train_"):
        return (
            "/content/drive/MyDrive/MicroPlastics/research_workspace/"
            "unified/wwtp/images/train/"
            f"{sample_id}.jpg"
        )

    if sample_id.startswith("wwtp_validation_"):
        return (
            "/content/drive/MyDrive/MicroPlastics/research_workspace/"
            "unified/wwtp/images/validation/"
            f"{sample_id}.jpg"
        )

    if sample_id.startswith("wwtp_test_"):
        return (
            "/content/drive/MyDrive/MicroPlastics/research_workspace/"
            "unified/wwtp/images/test/"
            f"{sample_id}.jpg"
        )

    raise ValueError(f"Unknown sample_id: {sample_id}")


def reconstruct_mask_path(sample_id):
    if sample_id.startswith("wwtp_train_"):
        return (
            "/content/drive/MyDrive/MicroPlastics/research_workspace/"
            "unified/wwtp/masks/train/"
            f"{sample_id}.png"
        )

    if sample_id.startswith("wwtp_validation_"):
        return (
            "/content/drive/MyDrive/MicroPlastics/research_workspace/"
            "unified/wwtp/masks/validation/"
            f"{sample_id}.png"
        )

    if sample_id.startswith("wwtp_test_"):
        return (
            "/content/drive/MyDrive/MicroPlastics/research_workspace/"
            "unified/wwtp/masks/test/"
            f"{sample_id}.png"
        )

    raise ValueError(f"Unknown sample_id: {sample_id}")


# ------------------------------------------------------------
# Re-run inference for selected images
#
# We use the SAME frozen epoch-48 checkpoint and SAME
# 512x512 / stride-384 tiling protocol.
# ------------------------------------------------------------

import torch
import segmentation_models_pytorch as smp

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/MicroPlastics/research_workspace/"
    "analysis/baseline_experiment/checkpoints/"
    "beach_unet_resnet50_best_dice.pth"
)

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

for param in model.parameters():
    param.requires_grad = False

print("\nCheckpoint epoch:", checkpoint.get("epoch"))
print("Best validation Dice:", checkpoint.get("best_val_dice"))
print("Model frozen:", all(not p.requires_grad for p in model.parameters()))


# ------------------------------------------------------------
# Tiling configuration — SAME as Step 14A / 14B
# ------------------------------------------------------------

TILE_SIZE = 512
STRIDE = 384
THRESHOLD = 0.5


def get_positions(length, tile_size=512, stride=384):
    positions = list(range(0, max(length - tile_size, 0) + 1, stride))

    final_pos = max(length - tile_size, 0)

    if positions[-1] != final_pos:
        positions.append(final_pos)

    return positions


def run_tiled_prediction(image_path):

    image = np.asarray(Image.open(image_path).convert("RGB"))
    H, W, _ = image.shape

    y_positions = get_positions(H, TILE_SIZE, STRIDE)
    x_positions = get_positions(W, TILE_SIZE, STRIDE)

    logits_sum = np.zeros((H, W), dtype=np.float32)
    counts = np.zeros((H, W), dtype=np.float32)

    with torch.no_grad():

        for y in y_positions:
            for x in x_positions:

                tile = image[
                    y:y + TILE_SIZE,
                    x:x + TILE_SIZE
                ]

                assert tile.shape == (
                    TILE_SIZE,
                    TILE_SIZE,
                    3
                ), tile.shape

                tensor = (
                    torch.from_numpy(tile)
                    .permute(2, 0, 1)
                    .float()
                    .unsqueeze(0)
                    / 255.0
                ).to(DEVICE)

                logits = model(tensor)

                logits = (
                    logits[0, 0]
                    .detach()
                    .cpu()
                    .numpy()
                )

                logits_sum[
                    y:y + TILE_SIZE,
                    x:x + TILE_SIZE
                ] += logits

                counts[
                    y:y + TILE_SIZE,
                    x:x + TILE_SIZE
                ] += 1.0

    assert np.all(counts > 0)

    stitched_logits = logits_sum / counts

    probability = 1.0 / (
        1.0 + np.exp(-stitched_logits)
    )

    prediction = (
        probability >= THRESHOLD
    ).astype(np.uint8)

    return image, prediction, probability


# ------------------------------------------------------------
# Generate qualitative panels
# ------------------------------------------------------------

panels = []

for _, row in selected.iterrows():

    sample_id = row["sample_id"]

    image_path = find_path(
        row,
        ["image_path", "image"]
    )

    mask_path = find_path(
        row,
        ["mask_path", "mask"]
    )

    if image_path is None:
        image_path = reconstruct_image_path(sample_id)

    if mask_path is None:
        mask_path = reconstruct_mask_path(sample_id)

    assert os.path.exists(image_path), image_path
    assert os.path.exists(mask_path), mask_path

    image, pred, probability = run_tiled_prediction(
        image_path
    )

    gt = np.asarray(
        Image.open(mask_path).convert("L")
    )

    gt = (gt > 0).astype(np.uint8)

    assert image.shape[:2] == gt.shape
    assert pred.shape == gt.shape

    panels.append({
        "sample_id": sample_id,
        "dice": float(row["dice"]),
        "iou": float(row["iou"]),
        "precision": float(row["precision"]),
        "recall": float(row["recall"]),
        "image": image,
        "gt": gt,
        "pred": pred,
        "probability": probability
    })

# ------------------------------------------------------------
# Contact sheet
# ------------------------------------------------------------

n_rows = len(panels)
n_cols = 4

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(16, 4 * n_rows)
)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for r, p in enumerate(panels):

    image = p["image"]
    gt = p["gt"]
    pred = p["pred"]

    # GT / prediction overlay:
    # Red = GT only
    # Blue = prediction only
    # Purple = overlap
    overlay = np.zeros_like(image)

    gt_only = (gt == 1) & (pred == 0)
    pred_only = (gt == 0) & (pred == 1)
    overlap = (gt == 1) & (pred == 1)

    overlay[gt_only] = [255, 0, 0]
    overlay[pred_only] = [0, 0, 255]
    overlay[overlap] = [255, 0, 255]

    axes[r, 0].imshow(image)
    axes[r, 0].set_title(
        f"{p['sample_id']}\n"
        f"Dice={p['dice']:.4f}, IoU={p['iou']:.4f}"
    )

    axes[r, 1].imshow(gt, cmap="gray")
    axes[r, 1].set_title("Ground Truth")

    axes[r, 2].imshow(pred, cmap="gray")
    axes[r, 2].set_title("Prediction")

    axes[r, 3].imshow(image)
    axes[r, 3].imshow(
        overlay,
        alpha=0.55
    )
    axes[r, 3].set_title(
        f"Overlay\n"
        f"P={p['precision']:.4f}, "
        f"R={p['recall']:.4f}"
    )

    for c in range(n_cols):
        axes[r, c].axis("off")

plt.tight_layout()

output_path = os.path.join(
    OUTPUT_DIR,
    "wwtp_baseline_qualitative_contact_sheet.png"
)

plt.savefig(
    output_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("\nSaved qualitative analysis:")
print(output_path)

print("\nSTEP 15A STATUS: COMPLETE")

Across the examples, the Beach-trained model is frequently producing large amounts of foreground prediction over the WWTP background. For example, wwtp_train_000078 has essentially no spatial overlap with the ground truth: Dice 0.0000, while the prediction covers a substantial region of the image. In wwtp_train_000163 and wwtp_train_000002, the same pattern is visible: the model detects extensive regions as foreground even though the annotated WWTP objects occupy only tiny areas.
The strongest evidence is the combination of the metrics and visual overlays:
- Precision is extremely low on many samples.
- Recall can remain high because the prediction is very large.
- Dice and IoU remain low because the predicted foreground does not spatially agree with the actual object masks.
- Some images, such as wwtp_train_000722, are detected reasonably well, with Dice 0.7464 and IoU 0.5954. So the model is not universally incapable of detecting WWTP objects; performance is highly variable across images.
- The qualitative predictions show extensive false-positive foreground regions, particularly in the textured/background areas.
This matches the aggregate result:
WWTP precision = 0.0393, recall = 0.4751, Dice = 0.0604.
So we can document the baseline failure as:
The Beach-trained baseline exhibits severe cross-domain degradation on WWTP imagery, characterized primarily by extensive false-positive foreground predictions and poor spatial agreement with the WWTP ground-truth masks. Performance is heterogeneous across WWTP images, with some images showing substantially better localization than others.

#Step 15B — Quantify the failure pattern

In [ ]:
# ============================================================
# STEP 15B — QUANTITATIVE WWTP FAILURE-PATTERN ANALYSIS
# ============================================================

import os
import json
import numpy as np
import pandas as pd

RESULTS_CSV = (
    "/content/drive/MyDrive/MicroPlastics/research_workspace/"
    "analysis/baseline_experiment/wwtp_results/"
    "wwtp_external_per_image_results.csv"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/MicroPlastics/research_workspace/"
    "analysis/baseline_experiment/step15_analysis"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(RESULTS_CSV)

# ------------------------------------------------------------
# Primary population only
# ------------------------------------------------------------

primary = df[df["primary_metric_eligible"] == True].copy()

assert len(primary) == 2004

# ------------------------------------------------------------
# Dice performance bands
# ------------------------------------------------------------

def dice_band(x):
    if x < 0.01:
        return "Dice < 0.01"
    elif x < 0.05:
        return "0.01 <= Dice < 0.05"
    elif x < 0.10:
        return "0.05 <= Dice < 0.10"
    elif x < 0.25:
        return "0.10 <= Dice < 0.25"
    elif x < 0.50:
        return "0.25 <= Dice < 0.50"
    else:
        return "Dice >= 0.50"

primary["dice_band"] = primary["dice"].apply(dice_band)

band_order = [
    "Dice < 0.01",
    "0.01 <= Dice < 0.05",
    "0.05 <= Dice < 0.10",
    "0.10 <= Dice < 0.25",
    "0.25 <= Dice < 0.50",
    "Dice >= 0.50"
]

band_summary = (
    primary
    .groupby("dice_band", observed=False)
    .agg(
        images=("sample_id", "count"),
        mean_dice=("dice", "mean"),
        mean_iou=("iou", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_pixel_accuracy=("pixel_accuracy", "mean")
    )
    .reindex(band_order)
)

band_summary["percentage"] = (
    100.0 * band_summary["images"] / len(primary)
)

print("=" * 75)
print("WWTP DICE PERFORMANCE BANDS")
print("=" * 75)
print(band_summary.round(4).to_string())

# ------------------------------------------------------------
# Prediction foreground fraction
#
# Calculate from TP + FP relative to total pixels.
# GT foreground fraction from TP + FN.
# ------------------------------------------------------------

total_pixels = (
    primary["height"] * primary["width"]
)

primary["predicted_positive_fraction"] = (
    primary["tp"] + primary["fp"]
) / total_pixels

primary["gt_positive_fraction"] = (
    primary["tp"] + primary["fn"]
) / total_pixels

primary["prediction_to_gt_area_ratio"] = np.where(
    primary["gt_positive_fraction"] > 0,
    primary["predicted_positive_fraction"]
    / primary["gt_positive_fraction"],
    np.nan
)

print("\n" + "=" * 75)
print("FOREGROUND AREA COMPARISON")
print("=" * 75)

print(
    f"GT foreground fraction:"
    f" mean={primary['gt_positive_fraction'].mean():.6f},"
    f" median={primary['gt_positive_fraction'].median():.6f}"
)

print(
    f"Predicted foreground fraction:"
    f" mean={primary['predicted_positive_fraction'].mean():.6f},"
    f" median={primary['predicted_positive_fraction'].median():.6f}"
)

valid_ratio = primary[
    np.isfinite(primary["prediction_to_gt_area_ratio"])
]

print(
    f"Prediction / GT foreground-area ratio:"
    f" mean={valid_ratio['prediction_to_gt_area_ratio'].mean():.2f}x,"
    f" median={valid_ratio['prediction_to_gt_area_ratio'].median():.2f}x"
)

# ------------------------------------------------------------
# Extreme false-positive cases
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("LARGEST PREDICTION / GT AREA RATIOS")
print("=" * 75)

largest_ratio = (
    valid_ratio
    .sort_values(
        "prediction_to_gt_area_ratio",
        ascending=False
    )
    .head(10)
)

print(
    largest_ratio[
        [
            "sample_id",
            "dice",
            "precision",
            "recall",
            "gt_positive_fraction",
            "predicted_positive_fraction",
            "prediction_to_gt_area_ratio"
        ]
    ].round(5).to_string(index=False)
)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

band_path = os.path.join(
    OUTPUT_DIR,
    "wwtp_dice_band_summary.csv"
)

band_summary.to_csv(band_path)

analysis_summary = {
    "primary_population": int(len(primary)),
    "mean_dice": float(primary["dice"].mean()),
    "median_dice": float(primary["dice"].median()),
    "mean_iou": float(primary["iou"].mean()),
    "median_iou": float(primary["iou"].median()),
    "mean_precision": float(primary["precision"].mean()),
    "median_precision": float(primary["precision"].median()),
    "mean_recall": float(primary["recall"].mean()),
    "median_recall": float(primary["recall"].median()),
    "mean_gt_foreground_fraction":
        float(primary["gt_positive_fraction"].mean()),
    "median_gt_foreground_fraction":
        float(primary["gt_positive_fraction"].median()),
    "mean_predicted_foreground_fraction":
        float(primary["predicted_positive_fraction"].mean()),
    "median_predicted_foreground_fraction":
        float(primary["predicted_positive_fraction"].median()),
    "mean_prediction_to_gt_area_ratio":
        float(valid_ratio["prediction_to_gt_area_ratio"].mean()),
    "median_prediction_to_gt_area_ratio":
        float(valid_ratio["prediction_to_gt_area_ratio"].median())
}

summary_path = os.path.join(
    OUTPUT_DIR,
    "wwtp_step15_quantitative_analysis.json"
)

with open(summary_path, "w") as f:
    json.dump(
        analysis_summary,
        f,
        indent=2
    )

print("\nSaved:")
print(band_path)
print(summary_path)

print("\nSTEP 15B STATUS: COMPLETE")

The Beach-trained U-Net–ResNet50 achieves strong in-domain Beach segmentation performance (Dice 0.8323) but exhibits severe degradation on the external WWTP domain (Dice 0.0604). Qualitative and quantitative analysis indicates that the dominant failure mode is excessive foreground prediction: the model predicts substantially more foreground area than is present in the WWTP ground truth, resulting in very low precision and poor spatial overlap despite moderate recall. The degradation is heterogeneous across WWTP images, with a small subset showing substantially better localization.

#Step 16 — Robust Augmentation Experiment
rotation: ±20° instead of ±10°
scale: 0.90–1.10 instead of 0.95–1.05
translation: ±10% instead of ±5%
brightness: 0.75–1.25
contrast: 0.75–1.25
saturation variation
hue variation
occasional Gaussian blur
occasional Gaussian noise
horizontal flip retained


In [ ]:
# ============================================================
# STEP 16A — DOMAIN-GENERALIZATION EXPERIMENT 1
# STRONGER ROBUST AUGMENTATION
#
# WWTP IS COMPLETELY UNSEEN.
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# ============================================================
# 2. PATHS
# ============================================================

MANIFEST_PATH = (
    "/content/drive/MyDrive/MicroPlastics/research_workspace/"
    "analysis/baseline_experiment/"
    "beach_baseline_grouped_manifest.csv"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/MicroPlastics/research_workspace/"
    "analysis/domain_generalization/"
    "experiment_01_strong_augmentation"
)

CHECKPOINT_DIR = os.path.join(
    OUTPUT_DIR,
    "checkpoints"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ============================================================
# 3. LOAD THE SAME GROUPED BEACH MANIFEST
# ============================================================

manifest = pd.read_csv(MANIFEST_PATH)

print("\nManifest shape:", manifest.shape)

# Only annotated-positive images are used for supervised
# train/validation/test, exactly as in the baseline.

train_df = manifest[
    (manifest["baseline_split"] == "train") &
    (manifest["baseline_role"] == "train")
].copy()

val_df = manifest[
    (manifest["baseline_split"] == "val") &
    (manifest["baseline_role"] == "val")
].copy()

test_df = manifest[
    (manifest["baseline_split"] == "test") &
    (manifest["baseline_role"] == "test_positive")
].copy()

assert len(train_df) == 790
assert len(val_df) == 169
assert len(test_df) == 169

print("\nBeach supervised populations:")
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

# ============================================================
# 4. DATASET
# ============================================================

class BeachSegmentationDataset(Dataset):

    def __init__(
        self,
        dataframe,
        training=False
    ):
        self.df = dataframe.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.df)

    def _strong_augmentation(self, image, mask):

        # ----------------------------------------------------
        # GEOMETRIC AUGMENTATION
        # ----------------------------------------------------

        # Horizontal flip
        if random.random() < 0.5:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

        # Rotation
        angle = random.uniform(-20.0, 20.0)

        image = image.rotate(
            angle,
            resample=Image.Resampling.BILINEAR,
            fillcolor=(0, 0, 0)
        )

        mask = mask.rotate(
            angle,
            resample=Image.Resampling.NEAREST,
            fillcolor=0
        )

        # ----------------------------------------------------
        # SCALE + CROP/PAD THROUGH RESIZE
        # ----------------------------------------------------

        scale = random.uniform(0.90, 1.10)

        new_size = max(
            1,
            int(round(512 * scale))
        )

        image = image.resize(
            (new_size, new_size),
            resample=Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (new_size, new_size),
            resample=Image.Resampling.NEAREST
        )

        # If smaller than 512, pad.
        # If larger than 512, random crop.

        if new_size < 512:

            canvas_img = Image.new(
                "RGB",
                (512, 512),
                (0, 0, 0)
            )

            canvas_mask = Image.new(
                "L",
                (512, 512),
                0
            )

            max_offset = 512 - new_size

            x = random.randint(0, max_offset)
            y = random.randint(0, max_offset)

            canvas_img.paste(
                image,
                (x, y)
            )

            canvas_mask.paste(
                mask,
                (x, y)
            )

            image = canvas_img
            mask = canvas_mask

        elif new_size > 512:

            max_offset = new_size - 512

            x = random.randint(0, max_offset)
            y = random.randint(0, max_offset)

            image = image.crop(
                (x, y, x + 512, y + 512)
            )

            mask = mask.crop(
                (x, y, x + 512, y + 512)
            )

        # ----------------------------------------------------
        # TRANSLATION
        # ----------------------------------------------------

        # Small translation implemented through a canvas.
        if random.random() < 0.5:

            dx = random.randint(
                -int(0.10 * 512),
                int(0.10 * 512)
            )

            dy = random.randint(
                -int(0.10 * 512),
                int(0.10 * 512)
            )

            translated_img = Image.new(
                "RGB",
                (512, 512),
                (0, 0, 0)
            )

            translated_mask = Image.new(
                "L",
                (512, 512),
                0
            )

            translated_img.paste(
                image,
                (dx, dy)
            )

            translated_mask.paste(
                mask,
                (dx, dy)
            )

            image = translated_img
            mask = translated_mask

        # ----------------------------------------------------
        # PHOTOMETRIC AUGMENTATION
        # ----------------------------------------------------

        # Brightness
        if random.random() < 0.8:

            factor = random.uniform(
                0.75,
                1.25
            )

            image = ImageEnhance.Brightness(
                image
            ).enhance(factor)

        # Contrast
        if random.random() < 0.8:

            factor = random.uniform(
                0.75,
                1.25
            )

            image = ImageEnhance.Contrast(
                image
            ).enhance(factor)

        # Saturation
        if random.random() < 0.5:

            factor = random.uniform(
                0.70,
                1.30
            )

            image = ImageEnhance.Color(
                image
            ).enhance(factor)

        # ----------------------------------------------------
        # BLUR
        # ----------------------------------------------------

        if random.random() < 0.20:

            radius = random.uniform(
                0.3,
                1.5
            )

            image = image.filter(
                ImageFilter.GaussianBlur(
                    radius=radius
                )
            )

        return image, mask

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        mask = Image.open(
            row["mask_path"]
        ).convert("L")

        # ----------------------------------------------------
        # ORIGINAL DATA = 256×256
        # BASELINE PIPELINE = 512×512
        # ----------------------------------------------------

        image = image.resize(
            (512, 512),
            resample=Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (512, 512),
            resample=Image.Resampling.NEAREST
        )

        if self.training:
            image, mask = self._strong_augmentation(
                image,
                mask
            )

        # ----------------------------------------------------
        # TENSOR CONVERSION
        # ----------------------------------------------------

        image_np = np.asarray(
            image,
            dtype=np.float32
        ).copy()

        mask_np = np.asarray(
            mask,
            dtype=np.uint8
        ).copy()

        image_tensor = (
            torch.from_numpy(image_np)
            .permute(2, 0, 1)
            .float()
            / 255.0
        )

        mask_tensor = torch.from_numpy(
            (mask_np > 0).astype(np.float32)
        ).unsqueeze(0)

        return image_tensor, mask_tensor


# ============================================================
# 5. DATALOADERS
# ============================================================

train_dataset = BeachSegmentationDataset(
    train_df,
    training=True
)

val_dataset = BeachSegmentationDataset(
    val_df,
    training=False
)

test_dataset = BeachSegmentationDataset(
    test_df,
    training=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nDataLoaders created.")

# ============================================================
# 6. VERIFY AUGMENTED BATCH
# ============================================================

images, masks = next(iter(train_loader))

print("\nAugmented batch:")
print("Images:", images.shape)
print("Masks :", masks.shape)
print("Image dtype:", images.dtype)
print("Mask dtype :", masks.dtype)
print("Image range:", images.min().item(), images.max().item())
print("Mask values:", torch.unique(masks))

assert images.shape == (8, 3, 512, 512)
assert masks.shape == (8, 1, 512, 512)

# ============================================================
# 7. MODEL — EXACT SAME ARCHITECTURE
# ============================================================

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(DEVICE)

print("\nModel initialized.")

print(
    "Total parameters:",
    sum(p.numel() for p in model.parameters())
)

# ============================================================
# 8. LOSS — EXACT SAME AS BASELINE
# ============================================================

bce_loss = nn.BCEWithLogitsLoss()

def dice_loss(logits, targets, smooth=1e-6):

    probabilities = torch.sigmoid(logits)

    probabilities = probabilities.reshape(
        probabilities.shape[0],
        -1
    )

    targets = targets.reshape(
        targets.shape[0],
        -1
    )

    intersection = (
        probabilities * targets
    ).sum(dim=1)

    dice = (
        2.0 * intersection + smooth
    ) / (
        probabilities.sum(dim=1)
        + targets.sum(dim=1)
        + smooth
    )

    return 1.0 - dice.mean()


def combined_loss(logits, targets):

    bce = bce_loss(
        logits,
        targets
    )

    dice = dice_loss(
        logits,
        targets
    )

    total = (
        0.5 * bce +
        0.5 * dice
    )

    return total


# ============================================================
# 9. METRICS
# ============================================================

def calculate_metrics(logits, targets):

    probabilities = torch.sigmoid(logits)

    predictions = (
        probabilities >= 0.5
    ).float()

    predictions = predictions.reshape(
        predictions.shape[0],
        -1
    )

    targets = targets.reshape(
        targets.shape[0],
        -1
    )

    tp = (
        predictions * targets
    ).sum(dim=1)

    fp = (
        predictions * (1 - targets)
    ).sum(dim=1)

    fn = (
        (1 - predictions) * targets
    ).sum(dim=1)

    tn = (
        (1 - predictions) * (1 - targets)
    ).sum(dim=1)

    dice = (
        2 * tp + 1e-7
    ) / (
        2 * tp + fp + fn + 1e-7
    )

    iou = (
        tp + 1e-7
    ) / (
        tp + fp + fn + 1e-7
    )

    precision = (
        tp + 1e-7
    ) / (
        tp + fp + 1e-7
    )

    recall = (
        tp + 1e-7
    ) / (
        tp + fn + 1e-7
    )

    accuracy = (
        tp + tn
    ) / (
        tp + tn + fp + fn + 1e-7
    )

    return {
        "dice": dice.mean().item(),
        "iou": iou.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item(),
        "pixel_accuracy": accuracy.mean().item()
    }


# ============================================================
# 10. OPTIMIZER — EXACT SAME AS BASELINE
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ============================================================
# 11. VALIDATION FUNCTION
# ============================================================

@torch.no_grad()
def evaluate_loader(loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    metric_sum = {
        "dice": 0.0,
        "iou": 0.0,
        "precision": 0.0,
        "recall": 0.0,
        "pixel_accuracy": 0.0
    }

    for images, masks in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        logits = model(images)

        loss = combined_loss(
            logits,
            masks
        )

        batch_size = images.shape[0]

        total_loss += (
            loss.item() * batch_size
        )

        metrics = calculate_metrics(
            logits,
            masks
        )

        for key in metric_sum:
            metric_sum[key] += (
                metrics[key] * batch_size
            )

        total_samples += batch_size

    results = {
        "loss": total_loss / total_samples
    }

    for key in metric_sum:
        results[key] = (
            metric_sum[key] /
            total_samples
        )

    return results


# ============================================================
# 12. TRAINING
# ============================================================

MAX_EPOCHS = 50
PATIENCE = 10

best_val_dice = -1.0
best_epoch = -1
patience_counter = 0

history = []

best_checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

full_checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "beach_unet_resnet50_strong_aug_full_checkpoint.pth"
)

print("\n" + "=" * 70)
print("STEP 16A — STRONG AUGMENTATION TRAINING")
print("=" * 70)

for epoch in range(1, MAX_EPOCHS + 1):

    model.train()

    running_loss = 0.0
    total_samples = 0

    for images, masks in train_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(images)

        loss = combined_loss(
            logits,
            masks
        )

        loss.backward()

        optimizer.step()

        batch_size = images.shape[0]

        running_loss += (
            loss.item() * batch_size
        )

        total_samples += batch_size

    train_loss = (
        running_loss /
        total_samples
    )

    val_results = evaluate_loader(
        val_loader
    )

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_results["loss"],
        "val_dice": val_results["dice"],
        "val_iou": val_results["iou"],
        "val_precision": val_results["precision"],
        "val_recall": val_results["recall"],
        "val_pixel_accuracy": val_results["pixel_accuracy"]
    }

    history.append(record)

    improved = (
        val_results["dice"] >
        best_val_dice
    )

    if improved:

        best_val_dice = (
            val_results["dice"]
        )

        best_epoch = epoch
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_dice": best_val_dice,
                "history": history,
                "experiment": "strong_augmentation",
                "seed": SEED
            },
            best_checkpoint_path
        )

    else:
        patience_counter += 1

    print(
        f"Epoch {epoch:02d} | "
        f"TrainLoss={train_loss:.4f} | "
        f"ValLoss={val_results['loss']:.4f} | "
        f"ValDice={val_results['dice']:.4f} | "
        f"ValIoU={val_results['iou']:.4f}"
        + (" | BEST" if improved else "")
    )

    if patience_counter >= PATIENCE:
        print(
            f"\nEarly stopping at epoch {epoch}."
        )
        break


# ============================================================
# 13. SAVE FULL CHECKPOINT
# ============================================================

torch.save(
    {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_dice": best_val_dice,
        "best_epoch": best_epoch,
        "history": history,
        "experiment": "strong_augmentation",
        "seed": SEED,
        "train_population": len(train_df),
        "validation_population": len(val_df),
        "test_population": len(test_df),
        "wwtp_used_for_training": False,
        "wwtp_used_for_validation": False,
        "wwtp_used_for_model_selection": False,
        "wwtp_reserved_for_external_evaluation": True
    },
    full_checkpoint_path
)

# ============================================================
# 14. SAVE HISTORY + CONFIG
# ============================================================

history_path = os.path.join(
    OUTPUT_DIR,
    "training_history.json"
)

with open(history_path, "w") as f:
    json.dump(
        history,
        f,
        indent=2
    )

config = {
    "experiment": "experiment_01_strong_augmentation",
    "task": "binary semantic segmentation",
    "architecture": "U-Net with ResNet50 ImageNet encoder",
    "input_size": "512x512x3",
    "output": "512x512x1 logits",
    "loss": "0.5 BCEWithLogits + 0.5 Dice",
    "optimizer": "AdamW",
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "batch_size": 8,
    "max_epochs": 50,
    "early_stopping_patience": 10,
    "seed": SEED,
    "augmentation": {
        "horizontal_flip": 0.5,
        "rotation_degrees": [-20, 20],
        "scale": [0.90, 1.10],
        "translation_fraction": 0.10,
        "brightness": [0.75, 1.25],
        "contrast": [0.75, 1.25],
        "saturation": [0.70, 1.30],
        "gaussian_blur_probability": 0.20,
        "gaussian_blur_radius": [0.3, 1.5]
    },
    "wwtp_seen_during_training": False,
    "wwtp_seen_during_validation": False,
    "wwtp_used_for_model_selection": False,
    "wwtp_external_evaluation": True
}

config_path = os.path.join(
    OUTPUT_DIR,
    "experiment_config.json"
)

with open(config_path, "w") as f:
    json.dump(
        config,
        f,
        indent=2
    )

print("\n" + "=" * 70)
print("STEP 16A TRAINING COMPLETE")
print("=" * 70)
print("Best epoch:", best_epoch)
print("Best validation Dice:", best_val_dice)
print("Best checkpoint:", best_checkpoint_path)
print("Full checkpoint:", full_checkpoint_path)
print("History:", history_path)
print("Config:", config_path)

print("\nIMPORTANT:")
print("WWTP was NOT used for training.")
print("WWTP was NOT used for validation.")
print("WWTP was NOT used for model selection.")
print("WWTP remains unseen for the later external evaluation.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

OUTPUT_DIR = (
    "/content/drive/MyDrive/MicroPlastics/research_workspace/"
    "analysis/domain_generalization/"
    "experiment_01_strong_augmentation"
)

CHECKPOINT_DIR = os.path.join(
    OUTPUT_DIR,
    "checkpoints"
)

print("Experiment directory exists:", os.path.exists(OUTPUT_DIR))
print("Checkpoint directory exists:", os.path.exists(CHECKPOINT_DIR))

if os.path.exists(OUTPUT_DIR):
    print("\nFiles already saved:")
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for f in files:
            path = os.path.join(root, f)
            size_mb = os.path.getsize(path) / (1024**2)
            print(f"{path}  |  {size_mb:.2f} MB")

Step 16A recovery — inspect checkpoint

In [ ]:
import torch
import os

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/MicroPlastics/research_workspace/"
    "analysis/domain_generalization/"
    "experiment_01_strong_augmentation/checkpoints/"
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("=" * 70)
print("STEP 16A CHECKPOINT INSPECTION")
print("=" * 70)

print("Checkpoint exists:", os.path.exists(CHECKPOINT_PATH))
print("Checkpoint size:",
      os.path.getsize(CHECKPOINT_PATH) / (1024**2),
      "MB")

print("\nCheckpoint keys:")
for key in checkpoint.keys():
    print(" -", key)

print("\nEpoch:", checkpoint.get("epoch"))
print("Best validation Dice:", checkpoint.get("best_val_dice"))
print("Experiment:", checkpoint.get("experiment"))
print("Seed:", checkpoint.get("seed"))

history = checkpoint.get("history")

if history is not None:
    print("\nHistory entries:", len(history))

    if len(history) > 0:
        print("\nLast recorded epoch:")
        print(history[-1])

        print("\nBest recorded epoch:")
        best_record = max(
            history,
            key=lambda x: x["val_dice"]
        )
        print(best_record)
else:
    print("\nNo history stored in checkpoint.")

print("\nSTEP 16A CHECKPOINT INSPECTION COMPLETE")

Step 16A — Resume from epoch 3

In [ ]:
# ============================================================
# STEP 16A — RESUME STRONG-AUGMENTATION TRAINING
# Resume from saved BEST checkpoint at epoch 3
# ============================================================

!pip install -q segmentation-models-pytorch

import os
import json
import random
import numpy as np
import pandas as pd

from PIL import Image, ImageEnhance, ImageFilter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


# ============================================================
# 2. PATHS
# ============================================================

MANIFEST_PATH = (
    "/content/drive/MyDrive/MicroPlastics/research_workspace/"
    "analysis/baseline_experiment/"
    "beach_baseline_grouped_manifest.csv"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/MicroPlastics/research_workspace/"
    "analysis/domain_generalization/"
    "experiment_01_strong_augmentation"
)

CHECKPOINT_DIR = os.path.join(
    OUTPUT_DIR,
    "checkpoints"
)

BEST_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

FULL_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "beach_unet_resnet50_strong_aug_full_checkpoint.pth"
)

HISTORY_PATH = os.path.join(
    OUTPUT_DIR,
    "training_history.json"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)


# ============================================================
# 3. LOAD MANIFEST
# ============================================================

manifest = pd.read_csv(
    MANIFEST_PATH
)

train_df = manifest[
    (manifest["baseline_split"] == "train") &
    (manifest["baseline_role"] == "train")
].copy()

val_df = manifest[
    (manifest["baseline_split"] == "val") &
    (manifest["baseline_role"] == "val")
].copy()

test_df = manifest[
    (manifest["baseline_split"] == "test") &
    (manifest["baseline_role"] == "test_positive")
].copy()

assert len(train_df) == 790
assert len(val_df) == 169
assert len(test_df) == 169

print("\nBeach populations:")
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))


# ============================================================
# 4. DATASET
# ============================================================

class BeachSegmentationDataset(Dataset):

    def __init__(self, dataframe, training=False):
        self.df = dataframe.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.df)

    def _strong_augmentation(self, image, mask):

        # -----------------------------
        # Horizontal flip
        # -----------------------------
        if random.random() < 0.5:
            image = image.transpose(
                Image.Transpose.FLIP_LEFT_RIGHT
            )
            mask = mask.transpose(
                Image.Transpose.FLIP_LEFT_RIGHT
            )

        # -----------------------------
        # Rotation
        # -----------------------------
        angle = random.uniform(
            -20.0,
            20.0
        )

        image = image.rotate(
            angle,
            resample=Image.Resampling.BILINEAR,
            fillcolor=(0, 0, 0)
        )

        mask = mask.rotate(
            angle,
            resample=Image.Resampling.NEAREST,
            fillcolor=0
        )

        # -----------------------------
        # Scale
        # -----------------------------
        scale = random.uniform(
            0.90,
            1.10
        )

        new_size = max(
            1,
            int(round(512 * scale))
        )

        image = image.resize(
            (new_size, new_size),
            resample=Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (new_size, new_size),
            resample=Image.Resampling.NEAREST
        )

        if new_size < 512:

            canvas_img = Image.new(
                "RGB",
                (512, 512),
                (0, 0, 0)
            )

            canvas_mask = Image.new(
                "L",
                (512, 512),
                0
            )

            max_offset = 512 - new_size

            x = random.randint(
                0,
                max_offset
            )

            y = random.randint(
                0,
                max_offset
            )

            canvas_img.paste(
                image,
                (x, y)
            )

            canvas_mask.paste(
                mask,
                (x, y)
            )

            image = canvas_img
            mask = canvas_mask

        elif new_size > 512:

            max_offset = new_size - 512

            x = random.randint(
                0,
                max_offset
            )

            y = random.randint(
                0,
                max_offset
            )

            image = image.crop(
                (x, y, x + 512, y + 512)
            )

            mask = mask.crop(
                (x, y, x + 512, y + 512)
            )

        # -----------------------------
        # Translation
        # -----------------------------
        if random.random() < 0.5:

            dx = random.randint(
                -int(0.10 * 512),
                int(0.10 * 512)
            )

            dy = random.randint(
                -int(0.10 * 512),
                int(0.10 * 512)
            )

            translated_img = Image.new(
                "RGB",
                (512, 512),
                (0, 0, 0)
            )

            translated_mask = Image.new(
                "L",
                (512, 512),
                0
            )

            translated_img.paste(
                image,
                (dx, dy)
            )

            translated_mask.paste(
                mask,
                (dx, dy)
            )

            image = translated_img
            mask = translated_mask

        # -----------------------------
        # Brightness
        # -----------------------------
        if random.random() < 0.8:

            factor = random.uniform(
                0.75,
                1.25
            )

            image = ImageEnhance.Brightness(
                image
            ).enhance(factor)

        # -----------------------------
        # Contrast
        # -----------------------------
        if random.random() < 0.8:

            factor = random.uniform(
                0.75,
                1.25
            )

            image = ImageEnhance.Contrast(
                image
            ).enhance(factor)

        # -----------------------------
        # Saturation
        # -----------------------------
        if random.random() < 0.5:

            factor = random.uniform(
                0.70,
                1.30
            )

            image = ImageEnhance.Color(
                image
            ).enhance(factor)

        # -----------------------------
        # Gaussian blur
        # -----------------------------
        if random.random() < 0.20:

            radius = random.uniform(
                0.3,
                1.5
            )

            image = image.filter(
                ImageFilter.GaussianBlur(
                    radius=radius
                )
            )

        return image, mask

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        mask = Image.open(
            row["mask_path"]
        ).convert("L")

        # Original Beach images: 256x256
        # Standard model input: 512x512

        image = image.resize(
            (512, 512),
            resample=Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (512, 512),
            resample=Image.Resampling.NEAREST
        )

        if self.training:
            image, mask = self._strong_augmentation(
                image,
                mask
            )

        image_np = np.asarray(
            image,
            dtype=np.float32
        ).copy()

        mask_np = np.asarray(
            mask,
            dtype=np.uint8
        ).copy()

        image_tensor = (
            torch.from_numpy(image_np)
            .permute(2, 0, 1)
            .float()
            / 255.0
        )

        mask_tensor = torch.from_numpy(
            (mask_np > 0).astype(np.float32)
        ).unsqueeze(0)

        return image_tensor, mask_tensor


# ============================================================
# 5. DATALOADERS
# ============================================================

train_dataset = BeachSegmentationDataset(
    train_df,
    training=True
)

val_dataset = BeachSegmentationDataset(
    val_df,
    training=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nDataLoaders ready.")


# ============================================================
# 6. MODEL
# ============================================================

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(DEVICE)


# ============================================================
# 7. LOSS
# ============================================================

bce_loss = nn.BCEWithLogitsLoss()


def dice_loss(logits, targets, smooth=1e-6):

    probabilities = torch.sigmoid(logits)

    probabilities = probabilities.reshape(
        probabilities.shape[0],
        -1
    )

    targets = targets.reshape(
        targets.shape[0],
        -1
    )

    intersection = (
        probabilities * targets
    ).sum(dim=1)

    dice = (
        2.0 * intersection + smooth
    ) / (
        probabilities.sum(dim=1)
        + targets.sum(dim=1)
        + smooth
    )

    return 1.0 - dice.mean()


def combined_loss(logits, targets):

    bce = bce_loss(
        logits,
        targets
    )

    dice = dice_loss(
        logits,
        targets
    )

    return (
        0.5 * bce +
        0.5 * dice
    )


# ============================================================
# 8. METRICS
# ============================================================

def calculate_metrics(logits, targets):

    probabilities = torch.sigmoid(logits)

    predictions = (
        probabilities >= 0.5
    ).float()

    predictions = predictions.reshape(
        predictions.shape[0],
        -1
    )

    targets = targets.reshape(
        targets.shape[0],
        -1
    )

    tp = (
        predictions * targets
    ).sum(dim=1)

    fp = (
        predictions * (1 - targets)
    ).sum(dim=1)

    fn = (
        (1 - predictions) * targets
    ).sum(dim=1)

    tn = (
        (1 - predictions) * (1 - targets)
    ).sum(dim=1)

    dice = (
        2 * tp + 1e-7
    ) / (
        2 * tp + fp + fn + 1e-7
    )

    iou = (
        tp + 1e-7
    ) / (
        tp + fp + fn + 1e-7
    )

    precision = (
        tp + 1e-7
    ) / (
        tp + fp + 1e-7
    )

    recall = (
        tp + 1e-7
    ) / (
        tp + fn + 1e-7
    )

    accuracy = (
        tp + tn
    ) / (
        tp + tn + fp + fn + 1e-7
    )

    return {
        "dice": dice.mean().item(),
        "iou": iou.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item(),
        "pixel_accuracy": accuracy.mean().item()
    }


# ============================================================
# 9. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ============================================================
# 10. LOAD SAVED CHECKPOINT
# ============================================================

print("\nLoading saved checkpoint...")

checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

history = checkpoint["history"]

best_val_dice = checkpoint["best_val_dice"]

start_epoch = checkpoint["epoch"] + 1

best_epoch = checkpoint["epoch"]

patience_counter = 0

print("Recovered epoch:", checkpoint["epoch"])
print("Recovered best Dice:", best_val_dice)
print("Continuing from epoch:", start_epoch)

assert checkpoint["experiment"] == "strong_augmentation"


# ============================================================
# 11. VALIDATION
# ============================================================

@torch.no_grad()
def evaluate_loader(loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    metric_sum = {
        "dice": 0.0,
        "iou": 0.0,
        "precision": 0.0,
        "recall": 0.0,
        "pixel_accuracy": 0.0
    }

    for images, masks in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        logits = model(images)

        loss = combined_loss(
            logits,
            masks
        )

        batch_size = images.shape[0]

        total_loss += (
            loss.item() * batch_size
        )

        metrics = calculate_metrics(
            logits,
            masks
        )

        for key in metric_sum:
            metric_sum[key] += (
                metrics[key] * batch_size
            )

        total_samples += batch_size

    results = {
        "loss": total_loss / total_samples
    }

    for key in metric_sum:
        results[key] = (
            metric_sum[key] / total_samples
        )

    return results


# ============================================================
# 12. CONTINUE TRAINING
# ============================================================

MAX_EPOCHS = 50
PATIENCE = 10

print("\n" + "=" * 70)
print("RESUMING STEP 16A")
print("=" * 70)

for epoch in range(start_epoch, MAX_EPOCHS + 1):

    model.train()

    running_loss = 0.0
    total_samples = 0

    for images, masks in train_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(images)

        loss = combined_loss(
            logits,
            masks
        )

        loss.backward()

        optimizer.step()

        batch_size = images.shape[0]

        running_loss += (
            loss.item() * batch_size
        )

        total_samples += batch_size

    train_loss = (
        running_loss /
        total_samples
    )

    val_results = evaluate_loader(
        val_loader
    )

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_results["loss"],
        "val_dice": val_results["dice"],
        "val_iou": val_results["iou"],
        "val_precision": val_results["precision"],
        "val_recall": val_results["recall"],
        "val_pixel_accuracy":
            val_results["pixel_accuracy"]
    }

    history.append(record)

    improved = (
        val_results["dice"] >
        best_val_dice
    )

    if improved:

        best_val_dice = (
            val_results["dice"]
        )

        best_epoch = epoch
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "best_val_dice":
                    best_val_dice,
                "history": history,
                "experiment":
                    "strong_augmentation",
                "seed": SEED
            },
            BEST_CHECKPOINT_PATH
        )

    else:
        patience_counter += 1

    print(
        f"Epoch {epoch:02d} | "
        f"TrainLoss={train_loss:.4f} | "
        f"ValLoss={val_results['loss']:.4f} | "
        f"ValDice={val_results['dice']:.4f} | "
        f"ValIoU={val_results['iou']:.4f}"
        + (" | BEST" if improved else "")
    )

    # --------------------------------------------------------
    # Save recovery checkpoint EVERY epoch.
    # This protects against another Colab disconnect.
    # --------------------------------------------------------

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict":
                model.state_dict(),
            "optimizer_state_dict":
                optimizer.state_dict(),
            "best_val_dice":
                best_val_dice,
            "best_epoch":
                best_epoch,
            "history":
                history,
            "experiment":
                "strong_augmentation",
            "seed":
                SEED
        },
        FULL_CHECKPOINT_PATH
    )

    with open(HISTORY_PATH, "w") as f:
        json.dump(
            history,
            f,
            indent=2
        )

    if patience_counter >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        break


# ============================================================
# 13. FINAL SAVE
# ============================================================

torch.save(
    {
        "epoch": epoch,
        "model_state_dict":
            model.state_dict(),
        "optimizer_state_dict":
            optimizer.state_dict(),
        "best_val_dice":
            best_val_dice,
        "best_epoch":
            best_epoch,
        "history":
            history,
        "experiment":
            "strong_augmentation",
        "seed":
            SEED,
        "wwtp_seen_during_training":
            False,
        "wwtp_seen_during_validation":
            False,
        "wwtp_used_for_model_selection":
            False,
        "wwtp_reserved_for_external_evaluation":
            True
    },
    FULL_CHECKPOINT_PATH
)

with open(HISTORY_PATH, "w") as f:
    json.dump(
        history,
        f,
        indent=2
    )


print("\n" + "=" * 70)
print("STEP 16A COMPLETE")
print("=" * 70)

print("Completed epoch:", epoch)
print("Best epoch:", best_epoch)
print("Best validation Dice:", best_val_dice)

print("\nBest checkpoint:")
print(BEST_CHECKPOINT_PATH)

print("\nRecovery checkpoint:")
print(FULL_CHECKPOINT_PATH)

print("\nTraining history:")
print(HISTORY_PATH)

print("\nWWTP was NOT used during training.")
print("WWTP was NOT used for validation.")
print("WWTP was NOT used for model selection.")

In [ ]:
import os

CHECKPOINT_DIR = (
    "/content/drive/MyDrive/MicroPlastics/research_workspace/"
    "analysis/domain_generalization/"
    "experiment_01_strong_augmentation/checkpoints"
)

for f in os.listdir(CHECKPOINT_DIR):
    path = os.path.join(CHECKPOINT_DIR, f)
    print(
        f,
        "|",
        round(os.path.getsize(path) / (1024**2), 2),
        "MB"
    )

#Tranfer to kaggle

In [ ]:
import os
import shutil
import pandas as pd
from pathlib import Path
import zipfile

# ---------------------------------------------------------
# STEP 1: Paths
# ---------------------------------------------------------

BASE = Path("/content/drive/MyDrive/MicroPlastics/research_workspace")

MANIFEST = BASE / "analysis/baseline_experiment/beach_baseline_grouped_manifest.csv"

STEP16A = BASE / "analysis/domain_generalization/experiment_01_strong_augmentation"

CHECKPOINT = STEP16A / "checkpoints/beach_unet_resnet50_strong_aug_best_dice.pth"

BUNDLE_DIR = STEP16A / "kaggle_bundle"

BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

print("Manifest:", MANIFEST)
print("Checkpoint:", CHECKPOINT)
print("Bundle:", BUNDLE_DIR)

print("\nManifest exists:", MANIFEST.exists())
print("Checkpoint exists:", CHECKPOINT.exists())

if CHECKPOINT.exists():
    print(
        "Checkpoint size:",
        round(CHECKPOINT.stat().st_size / (1024**2), 2),
        "MB"
    )

In [ ]:
import os
import shutil
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# STEP 2: Build portable Kaggle bundle
# ---------------------------------------------------------

BASE = Path("/content/drive/MyDrive/MicroPlastics/research_workspace")

MANIFEST = (
    BASE / "analysis/baseline_experiment/"
    "beach_baseline_grouped_manifest.csv"
)

STEP16A = (
    BASE / "analysis/domain_generalization/"
    "experiment_01_strong_augmentation"
)

CHECKPOINT = (
    STEP16A / "checkpoints/"
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

BUNDLE = STEP16A / "kaggle_bundle"

# Clean only the Kaggle bundle if it already exists.
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)

# Create directories
for split in ["train", "val", "test"]:
    (BUNDLE / "data" / split / "images").mkdir(parents=True)
    (BUNDLE / "data" / split / "masks").mkdir(parents=True)

(BUNDLE / "checkpoints").mkdir(parents=True)

# ---------------------------------------------------------
# Read manifest
# ---------------------------------------------------------

df = pd.read_csv(MANIFEST)

print("Original manifest:", df.shape)

# We only need supervised Beach samples.
# train = 790
# val   = 169
# test  = 169
#
# Empty-mask records are deliberately excluded.

keep_roles = ["train", "val", "test_positive"]

data = df[df["baseline_role"].isin(keep_roles)].copy()

print("\nSelected records:", len(data))
print(data["baseline_role"].value_counts())

# ---------------------------------------------------------
# Copy images and masks
# ---------------------------------------------------------

portable_rows = []

for idx, row in data.iterrows():

    role = row["baseline_role"]

    if role == "train":
        split = "train"
    elif role == "val":
        split = "val"
    elif role == "test_positive":
        split = "test"
    else:
        continue

    src_img = Path(row["image_path"])
    src_mask = Path(row["mask_path"])

    if not src_img.exists():
        raise FileNotFoundError(f"Image missing: {src_img}")

    if not src_mask.exists():
        raise FileNotFoundError(f"Mask missing: {src_mask}")

    # Preserve original filenames
    img_name = src_img.name
    mask_name = src_mask.name

    dst_img = BUNDLE / "data" / split / "images" / img_name
    dst_mask = BUNDLE / "data" / split / "masks" / mask_name

    shutil.copy2(src_img, dst_img)
    shutil.copy2(src_mask, dst_mask)

    # Make paths relative to Kaggle bundle
    new_row = row.copy()

    new_row["image_path"] = str(
        Path("data") / split / "images" / img_name
    )

    new_row["mask_path"] = str(
        Path("data") / split / "masks" / mask_name
    )

    new_row["portable_split"] = split

    portable_rows.append(new_row)

# ---------------------------------------------------------
# Save portable manifest
# ---------------------------------------------------------

portable_df = pd.DataFrame(portable_rows)

portable_manifest = BUNDLE / "portable_manifest.csv"

portable_df.to_csv(
    portable_manifest,
    index=False
)

# ---------------------------------------------------------
# Copy checkpoint
# ---------------------------------------------------------

portable_checkpoint = (
    BUNDLE / "checkpoints/"
    "beach_unet_resnet50_strong_aug_best_dice.pth"
)

shutil.copy2(
    CHECKPOINT,
    portable_checkpoint
)

# ---------------------------------------------------------
# Save a small README
# ---------------------------------------------------------

readme = BUNDLE / "README.txt"

with open(readme, "w") as f:
    f.write(
        "Step 16A - Strong Augmentation Domain Generalization\n\n"
        "Task: Beach supervised semantic segmentation\n"
        "Architecture: U-Net + ResNet-50 encoder\n"
        "Input: 512x512 RGB\n"
        "Output: 512x512 binary segmentation\n"
        "Loss: 0.5 BCEWithLogits + 0.5 Dice\n"
        "Optimizer: AdamW\n"
        "Checkpoint: epoch 3 recovery checkpoint\n"
        "WWTP: completely unseen / not included\n\n"
        "This package is intended for Kaggle GPU training.\n"
    )

# ---------------------------------------------------------
# Report
# ---------------------------------------------------------

print("\n==============================================")
print("KAGGLE BUNDLE CREATED")
print("==============================================")

print("\nManifest:")
print(portable_manifest)

print("\nCheckpoint:")
print(portable_checkpoint)

print("\nRecords:")
print(portable_df["portable_split"].value_counts())

print("\nFiles:")

for split in ["train", "val", "test"]:
    n_img = len(list(
        (BUNDLE / "data" / split / "images").glob("*")
    ))
    n_mask = len(list(
        (BUNDLE / "data" / split / "masks").glob("*")
    ))

    print(
        f"{split:5s} -> "
        f"{n_img} images, "
        f"{n_mask} masks"
    )

print(
    "\nCheckpoint size:",
    round(
        portable_checkpoint.stat().st_size / (1024**2),
        2
    ),
    "MB"
)

In [ ]:
import zipfile
from pathlib import Path

BUNDLE = Path(
    "/content/drive/MyDrive/MicroPlastics/"
    "research_workspace/analysis/domain_generalization/"
    "experiment_01_strong_augmentation/kaggle_bundle"
)

ZIP_PATH = BUNDLE.parent / "step16a_kaggle_bundle.zip"

print("Creating:", ZIP_PATH)

# Remove old archive if one exists
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

file_count = 0
total_bytes = 0

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_STORED
) as zf:

    for path in BUNDLE.rglob("*"):
        if path.is_file():

            # Keep "kaggle_bundle/..." as the archive root
            arcname = Path("kaggle_bundle") / path.relative_to(BUNDLE)

            zf.write(
                path,
                arcname=str(arcname)
            )

            file_count += 1
            total_bytes += path.stat().st_size

print("\n==============================================")
print("ZIP CREATED")
print("==============================================")

print("File count:", file_count)

print(
    "Uncompressed size:",
    round(total_bytes / (1024**3), 3),
    "GB"
)

print(
    "ZIP size:",
    round(ZIP_PATH.stat().st_size / (1024**3), 3),
    "GB"
)

print("\nZIP:")
print(ZIP_PATH)

In [ ]:
from pathlib import Path

BUNDLE = Path("/content/kaggle_microplastic_bundle")

print("Beach folder exists:", (BUNDLE / "data" / "beach").exists())

if (BUNDLE / "data" / "beach").exists():
    print("\nBeach contents:")
    for p in sorted((BUNDLE / "data" / "beach").iterdir()):
        print(" ", p.name)

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/MicroPlastics/research_workspace")

for base in [
    ROOT / "extracted" / "beach",
    ROOT / "unified" / "beach"
]:
    print("\n" + "=" * 80)
    print(base)
    print("=" * 80)

    if not base.exists():
        print("NOT FOUND")
        continue

    for p in sorted(base.iterdir()):
        print(("DIR " if p.is_dir() else "FILE"), p.name)

In [ ]:
from pathlib import Path

BEACH = Path(
    "/content/drive/MyDrive/MicroPlastics/research_workspace/extracted/beach"
)

for split in ["train", "val", "test"]:
    img_dir = BEACH / split / split / "images"
    mask_dir = BEACH / split / split / "labels"

    print(
        f"{split:5s} | "
        f"images = {len(list(img_dir.glob('*'))):4d} | "
        f"masks  = {len(list(mask_dir.glob('*'))):4d}"
    )

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

ROOT = Path("/content/drive/MyDrive/MicroPlastics/research_workspace")
SOURCE = ROOT / "extracted" / "beach"
MANIFEST = (
    ROOT
    / "analysis"
    / "baseline_experiment"
    / "beach_baseline_grouped_manifest.csv"
)
OUTPUT = Path("/content/beach_step16a")

# Start clean
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)

OUTPUT.mkdir(parents=True)

df = pd.read_csv(MANIFEST)

print("Manifest rows:", len(df))
print("\nBaseline split counts:")
print(df["baseline_split"].value_counts())

print("\nRole counts:")
print(df["baseline_role"].value_counts())


# ------------------------------------------------------------
# Locate an image/mask from the manifest with subdirectory safety
# ------------------------------------------------------------

def find_file(base, filename, subfolder_pattern):
    # Re-restrict the lookup to only match paths containing the correct subfolder (e.g. 'images' or 'labels')
    matches = [p for p in base.rglob(filename) if subfolder_pattern in p.parts]

    if len(matches) == 0:
        raise FileNotFoundError(f"Could not find: {filename} under subfolder containing '{subfolder_pattern}'")

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple files found for {filename}: {matches}"
        )

    return matches[0]


# ------------------------------------------------------------
# Reconstruct train / val / test
# ------------------------------------------------------------

for _, row in df.iterrows():

    split = row["baseline_split"]

    # Only the three actual model splits
    if split not in ["train", "val", "test"]:
        continue

    image_path = Path(row["image_path"])
    mask_path = Path(row["mask_path"])

    # Manifest paths may point to the original Drive location.
    # We restrict our searches to matching folders 'images' and 'labels'
    # respectively inside extracted/beach to prevent collision.
    image_src = find_file(SOURCE, image_path.name, "images")
    mask_src = find_file(SOURCE, mask_path.name, "labels")

    image_dst = OUTPUT / split / "images" / image_src.name
    mask_dst = OUTPUT / split / "masks" / mask_src.name

    image_dst.parent.mkdir(parents=True, exist_ok=True)
    mask_dst.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(image_src, image_dst)
    shutil.copy2(mask_src, mask_dst)


# ------------------------------------------------------------
# Verify reconstruction
# ------------------------------------------------------------

print("\n===========================================")
print("RECONSTRUCTED BEACH DATASET")
print("===========================================")

for split in ["train", "val", "test"]:

    images = list((OUTPUT / split / "images").glob("*"))
    masks = list((OUTPUT / split / "masks").glob("*"))

    print(
        f"{split:5s} | "
        f"images = {len(images):4d} | "
        f"masks = {len(masks):4d}"
    )

In [ ]:
import os

base = "/content/drive/MyDrive/MicroPlastics/research_workspace/analysis/domain_generalization/experiment_01_strong_augmentation"

print("Folder exists:", os.path.exists(base))
print("\nFiles inside:\n")

for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

In [ ]:

from pathlib import Path
import os

# Check whether Google Drive is mounted
drive_root = Path("/content/drive/MyDrive")

print("Drive accessible:", drive_root.exists())

# Locate the project directory
project_root = drive_root / "MicroPlastics"
print("Project exists:", project_root.exists())

# Check Beach data and checkpoint locations
paths_to_check = [
    project_root / "research_workspace/extracted/beach",
    project_root / "research_workspace/analysis",
    project_root / "research_workspace",
    project_root,
]

for path in paths_to_check:
    print(f"\n{path}")
    print("Exists:", path.exists())

    if path.exists() and path.is_dir():
        for item in list(path.iterdir())[:30]:
            print("  ", item.name, "(directory)" if item.is_dir() else "(file)")

# Search for model checkpoints and training histories
print("\n=== CHECKPOINTS AND TRAINING HISTORIES ===")
for root in [project_root]:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file() and (
                p.suffix.lower() in {".pth", ".pt", ".ckpt"}
                or "history" in p.name.lower()
            ):
                print(p)


In [ ]:

from pathlib import Path
import hashlib
import json
import torch

ROOT = Path("/content/drive/MyDrive/MicroPlastics/research_workspace")

paths = {
    "baseline_best": ROOT / "analysis/baseline_experiment/checkpoints/beach_unet_resnet50_best_dice.pth",
    "baseline_full": ROOT / "analysis/baseline_experiment/checkpoints/beach_unet_resnet50_baseline_full_checkpoint.pth",
    "strong_original": ROOT / "analysis/domain_generalization/experiment_01_strong_augmentation/checkpoints/beach_unet_resnet50_strong_aug_best_dice.pth",
    "strong_bundle": ROOT / "analysis/domain_generalization/experiment_01_strong_augmentation/kaggle_bundle/checkpoints/beach_unet_resnet50_strong_aug_best_dice.pth",
}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=== CHECKPOINT AUDIT ===")
for name, path in paths.items():
    print(f"\n{name}: {path}")
    print("Exists:", path.exists())
    if not path.exists():
        continue

    print("Size (MB):", round(path.stat().st_size / (1024**2), 2))
    print("SHA256:", sha256(path))

    obj = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(obj, dict):
        print("Keys:", list(obj.keys()))
        for key in ["epoch", "best_epoch", "best_val_dice", "experiment", "seed"]:
            if key in obj:
                print(f"{key}:", obj[key])

        history = obj.get("history")
        if isinstance(history, list):
            print("History length:", len(history))
            print("Last history records:", history[-3:])
        elif isinstance(history, dict):
            print("History keys:", list(history.keys()))

print("\n=== BEACH DIRECTORY STRUCTURE ===")
beach_root = ROOT / "extracted/beach"
for split in ["train", "val", "test"]:
    split_dir = beach_root / split
    print(f"\n{split}: {split_dir} | exists={split_dir.exists()}")
    if split_dir.exists():
        for p in list(split_dir.iterdir())[:15]:
            print(" ", p.name, "(directory)" if p.is_dir() else "(file)")

print("\n=== BASELINE HISTORY JSON ===")
history_path = ROOT / "analysis/baseline_experiment/beach_baseline_training_history.json"
if history_path.exists():
    with open(history_path, "r") as f:
        history = json.load(f)
    print("Type:", type(history).__name__)
    if isinstance(history, dict):
        print("Keys:", list(history.keys()))
        for key, value in history.items():
            if isinstance(value, list):
                print(f"{key}: length={len(value)}, last records={value[-3:]}")
            else:
                print(f"{key}: {value}")
    elif isinstance(history, list):
        print("Length:", len(history))
        print("Last records:", history[-3:])


In [ ]:

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/MicroPlastics/research_workspace")
beach_root = ROOT / "extracted/beach"

print("=== BEACH FILE STRUCTURE ===")
for split in ["train", "val", "test"]:
    split_root = beach_root / split
    print(f"\n[{split}]")
    for p in split_root.rglob("*"):
        if p.is_dir():
            print("DIR ", p.relative_to(beach_root))
        elif p.is_file():
            print("FILE", p.relative_to(beach_root))

        # Limit output if the directory contains many files
        if len(list(split_root.rglob("*"))) > 80:
            print("(output may be large; inspect folder names and first files)")
            break

print("\n=== BASELINE CHECKPOINTS ===")
for p in (ROOT / "analysis/baseline_experiment/checkpoints").glob("*"):
    if p.is_file():
        print(p.name, round(p.stat().st_size / (1024**2), 2), "MB")

print("\n=== MANIFESTS ===")
for name in [
    "beach_baseline_grouped_manifest.csv",
    "beach_segmentation_manifest.csv",
    "beach_canonical_manifest.csv",
    "microplastic_final_unified_manifest.csv",
]:
    for p in (ROOT / "analysis").rglob(name):
        print(p)


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/MicroPlastics/research_workspace")
beach_root = ROOT / "extracted/beach"

# 1. Show only directories and a few files per split
print("=== BEACH DIRECTORY CONTENTS ===")
for split in ["train", "val", "test"]:
    split_root = beach_root / split
    print(f"\n{split}:")
    if not split_root.exists():
        print("MISSING")
        continue

    for p in list(split_root.iterdir()):
        print("DIR " if p.is_dir() else "FILE", p.name)
        if p.is_dir():
            for child in list(p.iterdir())[:8]:
                print("   ", "DIR " if child.is_dir() else "FILE", child.name)
                if child.is_dir():
                    for f in list(child.iterdir())[:3]:
                        print("       ", f.name)

# 2. Inspect the grouped manifest
manifest = (
    ROOT / "analysis/baseline_experiment/beach_baseline_grouped_manifest.csv"
)
df = pd.read_csv(manifest)

print("\n=== GROUPED MANIFEST ===")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

for col in ["split", "baseline_split", "group_assignment", "baseline_role"]:
    if col in df.columns:
        print(f"\n{col} counts:")
        print(df[col].value_counts(dropna=False).to_string())

for col in ["image_path", "mask_path"]:
    if col in df.columns:
        print(f"\n{col} examples:")
        for value in df[col].dropna().head(5):
            p = Path(value)
            print(value, "| exists:", p.exists())

# 3. Confirm files exist under the actual data root
print("\n=== IMAGE/MASK FILE COUNTS ===")
for split in ["train", "val", "test"]:
    split_root = beach_root / split
    image_files = [
        p for p in split_root.rglob("*")
        if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"}
    ]
    print(split, "image files:", len(image_files))
    print("Examples:", [str(p.relative_to(beach_root)) for p in image_files[:5]])


In [ ]:

from pathlib import Path
import shutil
import json
import zipfile
import pandas as pd
import torch

ROOT = Path("/content/drive/MyDrive/MicroPlastics/research_workspace")
OUT = Path("/content/beach_diagnostic_bundle")

manifest_path = (
    ROOT / "analysis/baseline_experiment/beach_baseline_grouped_manifest.csv"
)
checkpoint_path = (
    ROOT / "analysis/baseline_experiment/checkpoints/beach_unet_resnet50_best_dice.pth"
)
history_path = (
    ROOT / "analysis/baseline_experiment/beach_baseline_training_history.json"
)

# Start with a clean staging directory
if OUT.exists():
    shutil.rmtree(OUT)

(OUT / "data/beach").mkdir(parents=True)
(OUT / "checkpoints").mkdir()
(OUT / "manifests").mkdir()
(OUT / "metadata").mkdir()

df = pd.read_csv(manifest_path)

required = ["image_path", "mask_path", "baseline_split"]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing manifest columns: {missing_columns}")

# Validate every manifest-listed source file before copying anything
problems = []
for idx, row in df.iterrows():
    img = Path(str(row["image_path"]))
    mask = Path(str(row["mask_path"]))

    if not img.is_file():
        problems.append((idx, "missing_image", str(img)))
    if not mask.is_file():
        problems.append((idx, "missing_mask", str(mask)))

if problems:
    print("Missing files found:", len(problems))
    print(*problems[:20], sep="\n")
    raise RuntimeError("Fix missing source files before packaging.")

allowed_splits = {"train", "val", "test"}
observed_splits = set(df["baseline_split"].dropna().astype(str))
if not observed_splits.issubset(allowed_splits):
    raise ValueError(f"Unexpected baseline_split values: {observed_splits}")

# Copy each manifest row using a unique filename.
# Original paths and split/eligibility fields remain in the manifest.
bundle_images = []
bundle_masks = []

for idx, row in df.iterrows():
    split = str(row["baseline_split"])
    item_id = f"beach_{idx:05d}"

    image_src = Path(str(row["image_path"]))
    mask_src = Path(str(row["mask_path"]))

    image_name = f"{item_id}{image_src.suffix.lower()}"
    mask_name = f"{item_id}{mask_src.suffix.lower()}"

    image_rel = Path("data/beach") / split / "images" / image_name
    mask_rel = Path("data/beach") / split / "masks" / mask_name

    image_dst = OUT / image_rel
    mask_dst = OUT / mask_rel
    image_dst.parent.mkdir(parents=True, exist_ok=True)
    mask_dst.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(image_src, image_dst)
    shutil.copy2(mask_src, mask_dst)

    bundle_images.append(str(image_rel))
    bundle_masks.append(str(mask_rel))

df["bundle_id"] = [f"beach_{i:05d}" for i in range(len(df))]
df["bundle_image_path"] = bundle_images
df["bundle_mask_path"] = bundle_masks

df.to_csv(OUT / "manifests/beach_baseline_grouped_manifest.csv", index=False)

# Copy the best baseline checkpoint and its training history
shutil.copy2(
    checkpoint_path,
    OUT / "checkpoints/beach_unet_resnet50_best_dice.pth"
)
shutil.copy2(
    history_path,
    OUT / "metadata/beach_baseline_training_history.json"
)

# Record checkpoint metadata for reference
ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
metadata = {
    "checkpoint_file": "checkpoints/beach_unet_resnet50_best_dice.pth",
    "epoch": ckpt.get("epoch"),
    "best_val_dice": ckpt.get("best_val_dice"),
    "val_iou": ckpt.get("val_iou"),
    "checkpoint_sha256_note": "Verify independently after transfer if needed.",
    "manifest_rows": len(df),
    "split_column_used_for_packaging": "baseline_split",
    "mask_pixels_modified": False,
}
with open(OUT / "metadata/bundle_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Write a short README
readme = """Beach Microplastic Diagnostic Bundle

The manifest-listed image/mask pairs are copied without changing pixel data.
The baseline_split column determines the packaged train/val/test directory.
Original manifest fields are retained; bundle_image_path and bundle_mask_path
give paths relative to this bundle.

Use evaluation_eligible and the documented role fields to decide which records
belong in supervised training or evaluation. Do not assume every record is a
positive annotated sample.

Checkpoint: best baseline model, selected at epoch 48.
This bundle does not contain the missing later strong-augmentation checkpoint.
"""
(OUT / "README.txt").write_text(readme)

# Verify packaged pairs and report split/role counts
print("=== PACKAGE CHECK ===")
print("Manifest rows:", len(df))
print("All source files existed before copy:", True)
print("\nRows by baseline_split:")
print(df["baseline_split"].value_counts(dropna=False).to_string())

for split in ["train", "val", "test"]:
    subset = df[df["baseline_split"] == split]
    print(
        f"{split}: rows={len(subset)}, "
        f"images={sum((OUT / p).is_file() for p in subset['bundle_image_path'])}, "
        f"masks={sum((OUT / p).is_file() for p in subset['bundle_mask_path'])}"
    )

print("\nBaseline roles:")
if "baseline_role" in df.columns:
    print(df["baseline_role"].value_counts(dropna=False).to_string())

# Zip the bundle
zip_path = Path("/content/drive/MyDrive/MicroPlastics/research_workspace/analysis/beach_diagnostic_bundle.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(Path("beach_diagnostic_bundle") / p.relative_to(OUT)))

print("\nZIP created:", zip_path)
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024**2), 2))


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/MicroPlastics")

print("=== LIKELY TRAINING NOTEBOOKS AND SCRIPTS ===")
extensions = {".ipynb", ".py"}

for p in PROJECT.rglob("*"):
    if not p.is_file() or p.suffix.lower() not in extensions:
        continue

    name = p.name.lower()
    path_text = str(p).lower()

    if any(term in name or term in path_text for term in [
        "baseline", "train", "experiment", "micro", "segment", "unet"
    ]):
        print(p)

print("\n=== BASELINE EXPERIMENT DIRECTORY ===")
baseline_dir = PROJECT / "research_workspace/analysis/baseline_experiment"

if baseline_dir.exists():
    for p in baseline_dir.rglob("*"):
        if p.is_file():
            print(p.relative_to(baseline_dir), f"({p.stat().st_size:,} bytes)")
else:
    print("Directory not found:", baseline_dir)

print("\n=== DOMAIN GENERALIZATION EXPERIMENT FILES ===")
dg_dir = PROJECT / "research_workspace/analysis/domain_generalization"

if dg_dir.exists():
    for p in dg_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".ipynb", ".py", ".json", ".txt"}:
            print(p.relative_to(dg_dir))


In [ ]:

from pathlib import Path
import json

BASE = Path(
    "/content/drive/MyDrive/MicroPlastics/"
    "research_workspace/analysis/baseline_experiment"
)

files = [
    "baseline_config.json",
    "beach_baseline_training_config.json",
    "wwtp_external_evaluation_config.json",
    "empty_mask_semantics.json",
    "step15_analysis/wwtp_step15_quantitative_analysis.json",
    "wwtp_results/wwtp_external_summary.json",
]

for rel in files:
    path = BASE / rel
    print("\n" + "=" * 80)
    print(rel)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    try:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        print(json.dumps(obj, indent=2))
    except Exception as e:
        print("Could not read JSON:", e)

print("\n=== PREVIOUS WWTP EVALUATION COLUMNS ===")
import pandas as pd

csv_path = BASE / "wwtp_results/wwtp_external_per_image_results.csv"
if csv_path.exists():
    results = pd.read_csv(csv_path)
    print("Shape:", results.shape)
    print("Columns:", results.columns.tolist())
    print(results.head(3).to_string(index=False))


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path(
    "/content/drive/MyDrive/MicroPlastics/"
    "research_workspace/analysis/baseline_experiment"
)

csv_path = BASE / "wwtp_results/wwtp_external_per_image_results.csv"
df = pd.read_csv(csv_path)

eligible = df[df["primary_metric_eligible"] == True].copy()

print("=== EVALUATION POPULATION ===")
print("All images:", len(df))
print("Primary-metric images:", len(eligible))
print("Excluded annotation-empty images:", len(df) - len(eligible))

print("\n=== PER-IMAGE METRIC DISTRIBUTIONS ===")
for col in [
    "dice", "iou", "precision", "recall",
    "pixel_accuracy", "tp", "fp", "fn"
]:
    if col in eligible.columns:
        print(f"\n{col}")
        print(eligible[col].describe(
            percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
        ).to_string())

# Reconstruct foreground fractions from confusion counts
pixels = eligible["height"] * eligible["width"]
eligible["gt_fg_fraction"] = (
    eligible["tp"] + eligible["fn"]
) / pixels
eligible["pred_fg_fraction"] = (
    eligible["tp"] + eligible["fp"]
) / pixels
eligible["pred_to_gt_area_ratio"] = (
    (eligible["tp"] + eligible["fp"]) /
    (eligible["tp"] + eligible["fn"]).clip(lower=1)
)

print("\n=== FOREGROUND AREA FRACTIONS ===")
print(eligible[
    ["gt_fg_fraction", "pred_fg_fraction", "pred_to_gt_area_ratio"]
].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_string())

print("\n=== WORST 15 IMAGES BY DICE ===")
cols = [
    "sample_id", "original_split", "dice", "precision", "recall",
    "gt_fg_fraction", "pred_fg_fraction", "pred_to_gt_area_ratio",
    "tp", "fp", "fn"
]
print(
    eligible.sort_values("dice")[cols]
    .head(15)
    .to_string(index=False)
)

print("\n=== BEST 15 IMAGES BY DICE ===")
print(
    eligible.sort_values("dice", ascending=False)[cols]
    .head(15)
    .to_string(index=False)
)

print("\n=== PERFORMANCE BY ORIGINAL DATASET SPLIT ===")
print(
    eligible.groupby("original_split")[
        ["dice", "precision", "recall", "gt_fg_fraction", "pred_fg_fraction"]
    ].agg(["count", "mean", "median"]).to_string()
)
